# Phase 2 — Hybrid RAG

Requires the provided Phase-1 Python export and saved Phase-1 outputs. BM25, RRF, context packing, citation parsing and gold-mask testing are implemented in this notebook.

```text
/kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/
├── nlp-phase1.py
├── course_pack_v1/course_pack_v1/{corpus.jsonl,questions_dev.jsonl,labels_dev.jsonl,questions_hidden.jsonl,score.py}
└── NLP_phase1_output/
    ├── document_embeddings_4B.npz   # optional: generated once when absent
    └── dev/dense/{predictions.jsonl,run_config.yaml}
```

Use Kaggle T4 x2 and Internet. Set the Git commit in CONFIG. Run all cells in order. `report.pdf` and the submission archive are generated from the run's results; the report is limited to three pages.


In [1]:
# =============================================================================
# §0-1  Environment setup: install dependencies, print versions, check GPU
# =============================================================================
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

_pip("-U", "transformers>=4.51.0", "accelerate", "pyyaml",
     "reportlab", "pypdf", "arabic_reshaper", "python-bidi", "rank_bm25")
print("dependencies ready")

import os, re, gc, json, math, time, random, string, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date

import numpy as np

import torch
import torch.nn.functional as F

# Enable PyTorch's expandable-segment allocator to reduce VRAM fragmentation,
# and turn on memory-efficient / flash scaled-dot-product attention when available.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if hasattr(torch.backends.cuda, "enable_sdp_backends"):
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_flash_sdp(True)

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import transformers

try:
    import accelerate
    ACCELERATE_VERSION = accelerate.__version__
except ImportError:
    ACCELERATE_VERSION = "NOT INSTALLED (required for device_map='auto')"

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("NumPy       :", np.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", ACCELERATE_VERSION)

assert torch.cuda.is_available(), (
    "A GPU is required. In Kaggle set Accelerator = GPU T4 x2."
)
print("Number of GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GiB")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.empty_cache()
gc.collect()


dependencies ready
Python      : 3.13.16
PyTorch     : 2.11.0+cu130
NumPy       : 2.1.3
transformers: 5.23.0
accelerate  : 1.16.0
Number of GPUs: 2
  GPU 0: Tesla T4 | 14.6 GiB
  GPU 1: Tesla T4 | 14.6 GiB


169

In [2]:
# =============================================================================
# §0-2  Central CONFIG — all paths, model choices and hyper-parameters live here
# =============================================================================
# Each run writes a `run_config.yaml` snapshot of this dict alongside predictions
# so every result is reproducible.

STD_ID       = "404722145"
STUDENT_NAME = "محمدحسین کریمی"
GITHUB_URL   = "https://github.com/mohammad-hossein-k-gh/Advanced-NLP_Phase1"
GITHUB_COMMIT = "5d7d445"  # Provided commit; verify it corresponds to the submitted Phase-2 code.
REPORT_DATE = "2026-10-23"  # User-provided report date, not automatically the execution date.

# -----------------------------------------------------------------------------
# Pinned Kaggle dataset path (must match the path Kaggle mounts your dataset at):
#   /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/
#       ├── course_pack_v1/{corpus.jsonl, questions_dev.jsonl, ...}
#       └── nlp-phase1.ipynb
# -----------------------------------------------------------------------------
KAGGLE_DATASET_ROOT = "/kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset"
PACK_SUBDIR         = "course_pack_v1"
PHASE1_NOTEBOOK     = "nlp-phase1.ipynb"

CONFIG = {
    # ---------- paths ----------
    "dataset_root": KAGGLE_DATASET_ROOT,
    # Alternate roots kept for compatibility with older Kaggle dataset slug layouts.
    # The pinned root above is always checked first.
    "dataset_root_alternates": [
        "/kaggle/input/nlp-phase1-dataset",
        "/kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset",
    ],
    "pack_subdir": PACK_SUBDIR,
    "phase1_notebook": PHASE1_NOTEBOOK,
    "phase1_outputs_dir": None,          # None = auto-detect Phase-1 artifacts
    "pack_dir": None,                    # override manually if needed; None = dataset_root/course_pack_v1
    "output_dir": "/kaggle/working/NLP_phase2",

    # ---------- models (kept identical to Phase 1) ----------
    "embedding_model": "Qwen/Qwen3-Embedding-4B",
    "llm_model": "Qwen/Qwen3-8B",
    "bge_reranker_model": "BAAI/bge-reranker-v2-m3",
    "qwen_reranker_model": "Qwen/Qwen3-Reranker-0.6B",
    # `max_memory` is passed to accelerate's `device_map="auto"`.
    # Keys MUST be integers (newer accelerate rejects string keys like "0"/"1").
    # Per-GPU cap is intentionally below total VRAM so headroom remains for
    # KV-cache activations during generation (the 8B generator needs ~1-2 GiB
    # of free VRAM per GPU for activations and KV-cache).
    "gpu_memory": {0: "12GiB", 1: "12GiB", "cpu": "40GiB"},

    # ---------- embedding ----------
    # Phase-1 document embedding call uses batch_size=4 and max_length=1024.
    # Capacity and truncation depend on actual data and hardware.
    "embed_batch": 4,
    "embed_max_len": 1024,
    # Phase-2 sentence-level experiment; tune batch size for available VRAM.
    "sent_batch": 16,
    "sent_max_len": 512,

    # ---------- retrieval ----------
    "retrieval_unit": "passage",   # "passage" or "sentence" (granularity experiment)
    # fusion_depth=100: each retriever returns 100 candidates before RRF.
    # Dense is a single matmul (cheap); BM25 over 3882 docs is also cheap.
    # 100 is a configurable candidate depth, not a measured optimum.
    "fusion_depth": 100,
    "k_output": 10,                # number of IDs written to `retrieved` (score.py KS=(2,5,10))
    "k_context": 10,               # cap on passages considered during packing
    "rrf_k": 60,                   # standard RRF constant (Cormack et al.; handout §9.1)
    "bm25_k1": 1.5,                # standard BM25 defaults (handout §9.1)
    "bm25_b": 0.75,
    "bm25_remove_stopwords": True,  # handout: remove stopwords before indexing

    # ---------- reranking ----------
    # rerank_candidates=30: reranker rescores the top-30 RRF candidates.
    # 30 is a cost/coverage choice, not a guaranteed optimum.
    "rerank_candidates": 30,
    "bge_rerank_batch": 16,
    "qwen_rerank_batch": 2,        # lower batch for causal-LM logits
    "rerank_max_length": 512,       # total tokens including the Qwen template

    # ---------- generation ----------
    # Start at batch 2; retry with smaller batches on CUDA out-of-memory.
    # Memory requirements and numerical effects depend on the runtime.
    "llm_batch": 2,
    "max_new_tokens": 96,          # Phase-2 generation limit; the Phase-1 function default was 128
    "max_prompt_tokens": 4096,     # Phase 1 truncation limit
    # Set a conservative context budget; measure its effect on this dataset.
    "max_context_tokens": 1000,
    "temperature": 0.0,            # greedy decoding (do_sample=False)

    # ---------- execution ----------
    "dev_limit": None,             # set to a small number (e.g. 20) for quick smoke runs only
    "run_dev": True,
    "run_dense_baseline": True,     # controlled dense comparison under the same Phase-2 generation settings
    "run_sentence": True,
    "run_rerank": True,
    "run_goldmask": True,
    "hidden_systems": ["hybrid"],
}

# Guard against accidentally using string GPU keys (older accelerate raises a
# cryptic "Device 0 is not recognized" error).
assert all(isinstance(k, int) or k in ("cpu", "disk", "mps")
           for k in CONFIG["gpu_memory"]), (
    "Keys in gpu_memory must be int (e.g. 0, 1); string keys like '0' are rejected by accelerate."
)

OUTPUT_DIR = Path(CONFIG["output_dir"])
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
RUNS_DIR = OUTPUT_DIR / "runs"
PROMPT_DIR = OUTPUT_DIR / "prompts"
for _d in (OUTPUT_DIR, ARTIFACT_DIR, RUNS_DIR, PROMPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print("Dataset root (pinned):", CONFIG["dataset_root"])
print("Pack subdir          :", CONFIG["pack_subdir"])
print("Phase-1 notebook     :", CONFIG["phase1_notebook"])
print("Output dir           :", OUTPUT_DIR)
print("\nCONFIG:")
for k, v in CONFIG.items():
    if k not in ("gpu_memory", "dataset_root_alternates"):
        print(f"  {k:22s} = {v}")


Dataset root (pinned): /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset
Pack subdir          : course_pack_v1
Phase-1 notebook     : nlp-phase1.ipynb
Output dir           : /kaggle/working/NLP_phase2

CONFIG:
  dataset_root           = /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset
  pack_subdir            = course_pack_v1
  phase1_notebook        = nlp-phase1.ipynb
  phase1_outputs_dir     = None
  pack_dir               = None
  output_dir             = /kaggle/working/NLP_phase2
  embedding_model        = Qwen/Qwen3-Embedding-4B
  llm_model              = Qwen/Qwen3-8B
  bge_reranker_model     = BAAI/bge-reranker-v2-m3
  qwen_reranker_model    = Qwen/Qwen3-Reranker-0.6B
  embed_batch            = 4
  embed_max_len          = 1024
  sent_batch             = 16
  sent_max_len           = 512
  retrieval_unit         = passage
  fusion_depth           = 100
  k_output               = 10
  k_context              = 10
  rrf_k                  = 60
  bm25_k1       

In [3]:
# =============================================================================
# §1  Resolve the dataset path and load course_pack_v1
# =============================================================================
PHASE2_ADAPTER_SOURCE = r"""
import json
import re
from pathlib import Path
import numpy as np
EMBEDDING_MODEL_NAME = 'Qwen/Qwen3-Embedding-4B'
LLM_MODEL_NAME = 'Qwen/Qwen3-8B'
TOP_K = 10
QUERY_INSTRUCT = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '

def build_doc_indexes(corpus):
    doc_ids, doc_texts = ([], [])
    docs_by_id, doc_title, doc_text, doc_sents = ({}, {}, {}, {})
    for item in corpus:
        doc_id = str(item['doc_id'])
        title = str(item.get('title', '')).strip()
        text = str(item.get('text', '')).strip()
        if not text:
            text = ' '.join(item.get('sentences', [])).strip()
        if not text:
            continue
        formatted = f'Title: {title}\nPassage: {text}'
        doc_ids.append(doc_id)
        doc_texts.append(formatted)
        docs_by_id[doc_id] = item
        doc_title[doc_id] = title
        doc_text[doc_id] = text
        doc_sents[doc_id] = list(item.get('sentences') or [])
    return (doc_ids, doc_texts, docs_by_id, doc_title, doc_text, doc_sents)

def query_texts_for(items, prefix=QUERY_INSTRUCT):
    return [prefix + item['question'] for item in items]

def dense_search_from_scores(scores, doc_ids, k=10, exclude_ids=frozenset()):
    s = np.asarray(scores, dtype=np.float32).copy()
    excl = {str(x) for x in exclude_ids}
    excl_rows = [i for i, d in enumerate(doc_ids) if str(d) in excl]
    if excl_rows:
        s[excl_rows] = -np.inf
    k_eff = max(1, min(k, len(doc_ids) - len(excl_rows)))
    order = np.lexsort((np.array([str(d) for d in doc_ids]), -s))
    return [str(doc_ids[i]) for i in order[:k_eff] if np.isfinite(s[i])]

def dense_search(query_vec, passage_emb, doc_ids, k=10, exclude_ids=frozenset()):
    sim = passage_emb @ np.asarray(query_vec, dtype=np.float32)
    return dense_search_from_scores(sim, doc_ids, k=k, exclude_ids=exclude_ids)
K_VALUES = [2, 5, 10]

def extract_short_answer(raw_text):
    text = re.sub('<think>.*?</think>', '', raw_text, flags=re.DOTALL)
    text = re.sub('\\s+', ' ', text).strip()
    matches = list(re.finditer('answer\\s*[:：]\\s*', text, flags=re.IGNORECASE))
    if matches:
        text = text[matches[-1].end():].strip()
    text = re.sub('\\[p\\d{5}\\]', '', text)
    text = re.sub('^(the answer is|therefore|so|thus|final answer)[,:\\s]*', '', text, flags=re.IGNORECASE).strip()
    text = text.strip('"\'“”«»')
    m = re.fullmatch('(p\\d{4,6})', text, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    if len(text.split()) > 10:
        text = re.split('(?<=[.!?])\\s+', text)[0]
    return re.sub('[\\s.,;:!?]+$', '', text).strip()
REQUIRED_FIELDS = {'qid', 'answer', 'retrieved', 'cited', 'abstained'}

def generate_from_prompts(prompts, tokenizer, model, max_new_tokens=96, max_length=4096, pad_token_id=None):
    import torch
    if pad_token_id is None:
        pad_token_id = getattr(tokenizer, 'pad_token_id', None)
        if pad_token_id is None:
            pad_token_id = getattr(tokenizer, 'eos_token_id', None)
    enc = tokenizer(prompts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    device = model.get_input_embeddings().weight.device
    enc = {k: v.to(device) for k, v in enc.items()}
    prompt_len = enc['input_ids'].shape[1]
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=pad_token_id, no_repeat_ngram_size=8, repetition_penalty=1.05)
    new_tokens = out[:, prompt_len:].detach().cpu()
    del out, enc
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

def run_score(pred_path, score_path, pack_dir, split='dev', compare=None, partial=False, validate_only=False):
    import subprocess
    import sys
    cmd = [sys.executable, str(score_path), str(pred_path), '--pack', str(pack_dir), '--split', split]
    if compare:
        cmd += ['--compare', str(compare)]
    if partial:
        cmd += ['--partial']
    if validate_only:
        cmd += ['--validate-only']
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f"score.py failed: {' '.join(cmd)}")
    return res.stdout

def parse_score_output(text):
    import re as _re
    overall, by_type, paired, type_n = ({}, {}, {}, {})
    overall_n, section, cur = (0, None, None)
    metric_re = _re.compile('^\\s+([A-Za-z0-9_@]+)\\s+([+-]?\\d+(?:\\.\\d+)?)\\s+\\[\\s*([+-]?\\d+(?:\\.\\d+)?),\\s*([+-]?\\d+(?:\\.\\d+)?)\\s*\\]')
    for line in text.splitlines():
        m = _re.match('^overall\\s*\\(n=(\\d+)\\)', line)
        if m:
            overall_n, section, cur = (int(m.group(1)), 'overall', None)
            continue
        m = _re.match('^type:\\s*(\\S+)\\s*\\(n=(\\d+)\\)', line)
        if m:
            section, cur = ('type', m.group(1))
            by_type[cur] = {}
            type_n[cur] = int(m.group(2))
            continue
        if line.startswith('paired difference'):
            section, cur = ('paired', None)
            continue
        m = metric_re.match(line)
        if m:
            key, val = (m.group(1), float(m.group(2)))
            lo, hi = (float(m.group(3)), float(m.group(4)))
            entry = {'value': val, 'ci': (lo, hi)}
            if section == 'overall':
                overall[key] = entry
            elif section == 'type':
                by_type[cur][key] = entry
            elif section == 'paired':
                paired[key] = entry
    return {'overall': overall, 'by_type': by_type, 'paired': paired, 'type_n': type_n, 'overall_n': overall_n}
"""

# Load only selected definitions from the user's Phase-1 Python export.
# Importing the entire export would rerun its model loading and experiments.
import ast as _ast
import types as _types
import torch.nn.functional as F

PHASE1_SOURCE_PATH = Path(CONFIG["dataset_root"]) / "nlp-phase1.py"
if not PHASE1_SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Required Phase-1 Python export not found: {PHASE1_SOURCE_PATH}")
_p1_source = PHASE1_SOURCE_PATH.read_text(encoding="utf-8")
_p1_tree = _ast.parse(_p1_source, filename=str(PHASE1_SOURCE_PATH))
for _source_key, _config_key in (("EMBEDDING_MODEL_NAME", "embedding_model"), ("LLM_MODEL_NAME", "llm_model")):
    _values = []
    for _node in _p1_tree.body:
        if isinstance(_node, _ast.Assign) and any(isinstance(t, _ast.Name) and t.id == _source_key for t in _node.targets):
            try:
                _values.append(_ast.literal_eval(_node.value))
            except (ValueError, TypeError):
                pass
    if not _values or any(v != CONFIG[_config_key] for v in _values):
        raise ValueError(f"Phase-1 {_source_key} does not match CONFIG[{_config_key!r}].")
_p1_names = {"read_jsonl", "make_embeddings", "retrieval_metrics", "validate_predictions", "save_predictions", "generate_answer", "dense_search"}
_p1_nodes = [n for n in _p1_tree.body if isinstance(n, _ast.FunctionDef) and n.name in _p1_names]
_missing = _p1_names - {n.name for n in _p1_nodes}
if _missing:
    raise RuntimeError(f"Phase-1 export is missing required definitions: {sorted(_missing)}")
PHASE1_UTILS = _types.ModuleType("phase1_runtime")
# Phase-2-specific adapters are local; selected Phase-1 functions come from the
# user-provided script itself, without executing the script's top-level cells.
exec(compile(PHASE2_ADAPTER_SOURCE, "<phase2-adapters>", "exec"), PHASE1_UTILS.__dict__)
_p1_globals = PHASE1_UTILS.__dict__
_p1_globals.update({"torch": torch, "F": F, "np": np, "json": json, "re": re, "Path": Path})
exec(compile(_ast.Module(body=_p1_nodes, type_ignores=[]), str(PHASE1_SOURCE_PATH), "exec"), _p1_globals)
# Recover the query prefix from the original query list comprehension.
_query_assignments = [n for n in _p1_tree.body if isinstance(n, _ast.Assign)
                      and any(isinstance(t, _ast.Name) and t.id == "query_texts" for t in n.targets)
                      and isinstance(n.value, _ast.ListComp)]
if not _query_assignments or not isinstance(_query_assignments[0].value.elt, _ast.BinOp):
    raise RuntimeError("Could not locate the Phase-1 query instruction in the Python export.")
try:
    _query_prefix = _ast.literal_eval(_query_assignments[0].value.elt.left)
except (ValueError, TypeError) as exc:
    raise RuntimeError("Could not safely read the Phase-1 query instruction.") from exc
PHASE1_UTILS.QUERY_INSTRUCT = _query_prefix
def _query_texts_for(items, prefix=None):
    prefix = _query_prefix if prefix is None else prefix
    return [prefix + item["question"] for item in items]
PHASE1_UTILS.query_texts_for = _query_texts_for
_p1_dense = PHASE1_UTILS.dense_search
def _dense_from_scores(scores, doc_ids, k=10, exclude_ids=frozenset()):
    _p1_globals["question_text_index"] = {"__phase2_query__": 0}
    _p1_globals["similarities"] = np.asarray(scores)[None, :]
    _p1_globals["saved_doc_ids"] = [str(x) for x in doc_ids]
    _p1_globals["doc_id_to_index"] = {str(d): i for i, d in enumerate(doc_ids)}
    return _p1_dense("__phase2_query__", k=k, exclude_ids=exclude_ids)
PHASE1_UTILS.dense_search_from_scores = _dense_from_scores
_p1_make_embeddings = PHASE1_UTILS.make_embeddings
def _phase1_embeddings(texts, tokenizer, model, batch_size=4, max_length=1024):
    _p1_globals["tokenizer"] = tokenizer
    _p1_globals["model"] = model
    return _p1_make_embeddings(texts, batch_size=batch_size, max_length=max_length)
PHASE1_UTILS.make_embeddings = _phase1_embeddings
print(f"Selected Phase-1 definitions loaded from: {PHASE1_SOURCE_PATH}")
read_jsonl = PHASE1_UTILS.read_jsonl
build_doc_indexes = PHASE1_UTILS.build_doc_indexes
query_texts_for = PHASE1_UTILS.query_texts_for
make_embeddings = PHASE1_UTILS.make_embeddings
dense_search_from_scores = PHASE1_UTILS.dense_search_from_scores
retrieval_metrics = PHASE1_UTILS.retrieval_metrics
extract_short_answer = PHASE1_UTILS.extract_short_answer
validate_predictions = PHASE1_UTILS.validate_predictions
save_predictions = PHASE1_UTILS.save_predictions
generate_from_prompts = PHASE1_UTILS.generate_from_prompts
generate_answer = PHASE1_UTILS.generate_answer
run_score = PHASE1_UTILS.run_score
parse_score_output = PHASE1_UTILS.parse_score_output
REQUIRED_FIELDS = PHASE1_UTILS.REQUIRED_FIELDS
QUERY_INSTRUCT = PHASE1_UTILS.QUERY_INSTRUCT

# Required files inside the course pack.
PACK_FILES = ["corpus.jsonl", "questions_dev.jsonl",
              "labels_dev.jsonl", "questions_hidden.jsonl"]
OPTIONAL_FILES = ["score.py", "README.txt", "kg_subset_passages.txt",
                  "LICENSE_AND_ATTRIBUTION.txt"]


def _candidate_roots():
    """Return candidate dataset roots in priority order.

    The pinned path from CONFIG is always first. All returned paths exist.
    """
    roots = [Path(CONFIG["dataset_root"])]
    for alt in CONFIG.get("dataset_root_alternates", []):
        roots.append(Path(alt))
    roots += [Path("/kaggle/input"), Path("/kaggle/working"),
              Path.cwd(), Path("/home/user")]
    seen, out = set(), []
    for r in roots:
        try:
            rp = r.resolve()
        except OSError:
            continue
        if rp in seen or not rp.exists():
            continue
        seen.add(rp)
        out.append(rp)
    return out


def _pack_is_complete(d: Path) -> bool:
    """Return True if directory *d* contains all required pack files."""
    return all((d / f).is_file() for f in PACK_FILES)


def resolve_dataset_root():
    """Find the dataset root that contains course_pack_v1.

    The pinned root is checked first; if that fails, alternates are tried; only
    if none match do we fall back to an rglob search (with a warning), and if
    that also fails we raise a clear error.
    """
    # 1) exact pinned root and its alternates
    for root in [Path(CONFIG["dataset_root"])] + [Path(a) for a in CONFIG["dataset_root_alternates"]]:
        if not root.exists():
            continue
        for sub in (root / CONFIG["pack_subdir"] / CONFIG["pack_subdir"], root / CONFIG["pack_subdir"], root):
            if sub.is_dir() and _pack_is_complete(sub):
                print(f"dataset root resolved: {root}")
                return root

    # 2) fallback rglob on /kaggle/input (warn the user because this is not the pinned layout)
    search_roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    search_roots += [Path.cwd(), Path("/home/user")]
    for sr in search_roots:
        try:
            hits = list(sr.rglob("corpus.jsonl"))
        except OSError:
            hits = []
        for p in hits:
            d = p.parent
            if _pack_is_complete(d):
                print(f"WARNING: dataset found via rglob fallback: {d} "
                      f"(expected {CONFIG['dataset_root']})")
                return d.parent if d.name == CONFIG["pack_subdir"] else d

    raise FileNotFoundError(
        "Could not find the course pack. Expected layout:\n"
        f"  {CONFIG['dataset_root']}/{CONFIG['pack_subdir']}/{{{','.join(PACK_FILES)}}}\n"
        "Make sure the dataset is attached via Kaggle → Add Data."
    )


DATASET_ROOT = resolve_dataset_root()

# List dataset contents (mirrors Phase 1's /kaggle/input walk but scoped to our dataset).
print("\nDataset contents:")
_shown = 0
for _root, _dirs, _files in os.walk(DATASET_ROOT):
    for _f in sorted(_files):
        rp = Path(_root) / _f
        try:
            rel = rp.relative_to(DATASET_ROOT)
        except ValueError:
            rel = rp
        print(" ", rel)
        _shown += 1
        if _shown >= 80:
            print("  ... (truncated)")
            break
    if _shown >= 80:
        break


def find_pack_dir():
    """Return the directory that contains the JSONL pack files."""
    override = CONFIG.get("pack_dir")
    if override:
        d = Path(override)
        if not _pack_is_complete(d):
            raise FileNotFoundError(f"Configured pack_dir {d} does not contain all required files.")
        return d

    primary = DATASET_ROOT / CONFIG["pack_subdir"] / CONFIG["pack_subdir"]
    if _pack_is_complete(primary):
        return primary
    primary = DATASET_ROOT / CONFIG["pack_subdir"]
    if _pack_is_complete(primary):
        return primary

    # Files are loose at the dataset root (non-standard layout).
    if _pack_is_complete(DATASET_ROOT):
        print(f"WARNING: pack files are at dataset root rather than {CONFIG['pack_subdir']}/")
        return DATASET_ROOT

    missing = [f for f in PACK_FILES if not (DATASET_ROOT / f).is_file()
               and not (primary / f).is_file()]
    raise FileNotFoundError(
        f"Course pack not found under {primary}. Missing files: {missing}"
    )


PACK_DIR = find_pack_dir()
print("\nPack dir:", PACK_DIR)
for f in PACK_FILES + OPTIONAL_FILES:
    print(f"  {'OK ' if (PACK_DIR / f).exists() else '--'} {f}")


def resolve_score_script():
    """Locate score.py in the pack, dataset root, or anywhere under /kaggle/input."""
    candidates = [PACK_DIR / "score.py", DATASET_ROOT / "score.py"]
    search_roots = [Path("/kaggle/input")] if Path("/kaggle/input").exists() else []
    search_roots += [Path.cwd(), Path("/home/user")]
    for sr in search_roots:
        try:
            candidates += list(sr.rglob("score.py"))
        except OSError:
            pass
    for c in candidates:
        if c.is_file():
            return c
    raise FileNotFoundError(
        "score.py not found. It should be inside the course pack or at the dataset root."
    )


SCORE_PATH = resolve_score_script()
print("\nscore.py:", SCORE_PATH)


corpus_path         = PACK_DIR / "corpus.jsonl"
questions_dev_path  = PACK_DIR / "questions_dev.jsonl"
labels_dev_path     = PACK_DIR / "labels_dev.jsonl"
questions_hidden_path = PACK_DIR / "questions_hidden.jsonl"

corpus            = read_jsonl(corpus_path)
questions_dev     = read_jsonl(questions_dev_path)
labels_dev        = read_jsonl(labels_dev_path)
questions_hidden  = read_jsonl(questions_hidden_path)

print("\nLoaded:")
print("  corpus          :", len(corpus), "passages  -", corpus_path)
print("  questions_dev   :", len(questions_dev), "questions -", questions_dev_path)
print("  labels_dev      :", len(labels_dev), "labels    -", labels_dev_path)
print("  questions_hidden:", len(questions_hidden), "questions -", questions_hidden_path)

# Fast lookup structures — built with the Phase-1 build_doc_indexes function.
DOC_IDS, _doc_texts, DOCS_BY_ID, DOC_TITLE, DOC_TEXT, DOC_SENTS = \
    build_doc_indexes(corpus)
# Normalize sentences the same way Phase 2 expects (stripped, non-empty).
DOC_SENTS = {d: [str(s).strip() for s in (v or []) if str(s).strip()]
             for d, v in DOC_SENTS.items()}
DOC_IDX = {d: i for i, d in enumerate(DOC_IDS)}
assert len(DOC_IDS) == len(set(DOC_IDS)), "duplicate doc_ids in corpus"
print(f"doc indexes built using the Phase-2 adapter ({len(DOC_IDS)} docs)")

LABELS_BY_QID = {str(l["qid"]): l for l in labels_dev}
QDEV_BY_QID   = {str(q["qid"]): q for q in questions_dev}

# Structural sanity checks.
assert len(set(DOC_IDS)) == len(DOC_IDS), "duplicate doc_ids in corpus"
assert set(LABELS_BY_QID) == set(QDEV_BY_QID), "qid mismatch between dev questions and labels"
assert set(str(q["qid"]) for q in questions_dev).isdisjoint(
    set(str(q["qid"]) for q in questions_hidden)), "dev and hidden qids overlap"


def passage_text(doc_id):
    """Format a passage identically to Phase 1 (critical for embedding reuse)."""
    return f"Title: {DOC_TITLE[doc_id]}\nPassage: {DOC_TEXT[doc_id]}"


# PASSAGE_TEXTS: when Phase-1 code is imported, use the exact Phase-1 formatted
# texts produced by build_doc_indexes (guarantees identical embeddings);
# otherwise rebuild them from the same dicts.
PASSAGE_TEXTS = _doc_texts

# Aggregate statistics.
from collections import Counter as _Counter
_type_counts = _Counter(str(q.get("type", "?")) for q in questions_dev)
_gold_counts = [len(l.get("gold_doc_ids", [])) for l in labels_dev]
print("\nDev question types:", dict(_type_counts))
print(f"Gold docs per question: mean={np.mean(_gold_counts):.2f} "
      f"min={min(_gold_counts)} max={max(_gold_counts)}")
_q0 = questions_dev[0]
print("\nSample dev question:")
print("  qid    :", _q0["qid"])
print("  type   :", _q0.get("type", "?"))
print("  question:", _q0["question"])
_l0 = LABELS_BY_QID[str(_q0["qid"])]
print("  gold   :", _l0.get("gold_doc_ids"), "answer:", _l0.get("answer"))


Selected Phase-1 definitions loaded from: /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/nlp-phase1.py
dataset root resolved: /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset

Dataset contents:
  nlp-phase1.py
  NLP_phase1_output/dev/closed_book/predictions.jsonl
  NLP_phase1_output/dev/closed_book/run_config.yaml
  NLP_phase1_output/dev/dense/predictions.jsonl
  NLP_phase1_output/dev/dense/run_config.yaml
  NLP_phase1_output/dev/oracle/predictions.jsonl
  NLP_phase1_output/dev/oracle/run_config.yaml
  NLP_phase1_output/hidden/closed_book/predictions.jsonl
  NLP_phase1_output/hidden/closed_book/run_config.yaml
  NLP_phase1_output/hidden/dense/predictions.jsonl
  NLP_phase1_output/hidden/dense/run_config.yaml
  NLP_phase1_output/selfcheck/predictions.jsonl
  course_pack_v1/course_pack_v1/LICENSE_AND_ATTRIBUTION.txt
  course_pack_v1/course_pack_v1/README.txt
  course_pack_v1/course_pack_v1/corpus.jsonl
  course_pack_v1/course_pack_v1/example_predict.py
  course_pa

## §1.5 Phase-1 artifacts

The original Python export supplies selected functions and the query instruction without running its top-level experiments. The Phase-1 dense dev run is mandatory. When saved document embeddings are absent, the notebook creates them using the Phase-1 function and saves them for reuse.


In [4]:
# =============================================================================
# §1.5  Audit the Phase 1 notebook and auto-discover its artifacts
# =============================================================================
import ast as _ast

# The query instruction was read from the required Python export.
DEFAULT_QUERY_INSTRUCT = QUERY_INSTRUCT

PHASE1_NB_PATH = DATASET_ROOT / CONFIG["phase1_notebook"]
P1_EMBED_NAME = P1_LLM_NAME = None
P1_STD_ID = P1_GH_URL = P1_GH_COMMIT = P1_OUT_DIR = P1_TOP_K = None
PHASE1_SRC = ""

if PHASE1_NB_PATH.exists():
    nb = json.loads(PHASE1_NB_PATH.read_text(encoding="utf-8"))
    parts = []
    for c in nb.get("cells", []):
        s = c.get("source", "")
        if isinstance(s, list):
            s = "".join(s)
        parts.append(s)
    PHASE1_SRC = "\n\n".join(parts)

    # Pull out a handful of well-known Phase-1 constants via regex + literal_eval.
    targets = {
        "EMBEDDING_MODEL_NAME": "P1_EMBED_NAME",
        "LLM_MODEL_NAME": "P1_LLM_NAME",
        "STD_ID": "P1_STD_ID",
        "GITHUB_URL": "P1_GH_URL",
        "GITHUB_COMMIT": "P1_GH_COMMIT",
        "OUTPUT_DIR": "P1_OUT_DIR",
        "TOP_K": "P1_TOP_K",
    }
    for src_var, dst in targets.items():
        m = re.search(rf"^{src_var}\s*=\s*(.+)$", PHASE1_SRC, flags=re.MULTILINE)
        if not m:
            continue
        raw = m.group(1).strip().rstrip("\n")
        try:
            globals()[dst] = _ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            globals()[dst] = raw.strip().strip("\"'")
        print(f"Phase-1 {src_var:20s} =", globals()[dst])

    # Extract the two-string query prefix pattern used in Phase 1 and sanity-check it.
    m1 = re.search(r'"Instruct: Given a web search query, retrieve relevant passages that answer the query"\s*\\n\s*"Query: "', PHASE1_SRC)
    if m1:
        try:
            extracted = _ast.literal_eval('(' + m1.group(0).replace('\\n', '"+"\\n"+"') + ')')
            extracted = "".join(extracted) if isinstance(extracted, tuple) else extracted
        except Exception:
            extracted = DEFAULT_QUERY_INSTRUCT
        QUERY_INSTRUCT = extracted
        assert QUERY_INSTRUCT == DEFAULT_QUERY_INSTRUCT, (
            f"Phase-1 query prefix differs from Phase-2 default; embeddings would not match.\n"
            f"  phase1: {QUERY_INSTRUCT!r}\n  phase2: {DEFAULT_QUERY_INSTRUCT!r}"
        )
        print("Phase-1 query prefix: matched default (embeddings are compatible).")

    # The Python export is authoritative; do not override its settings from an older notebook.
    # Print where each Phase-1 mechanism lives (useful for the oral defense).
    _markers = {
        "make_embeddings": "embedding function",
        "dense_search":    "dense retrieval",
        "pack_context":    "context packing (if any)",
        "generate_batch":  "LLM generation",
        "parse_llm_output":"answer parsing",
        "BM25":            "BM25 (if hand-written in Phase 1)",
        "RUN_PATHS":       "run paths",
    }
    print("\nPhase-1 notebook markers:")
    for i, c in enumerate(nb.get("cells", [])):
        s = "".join(c.get("source", [])) if isinstance(c.get("source", []), list) else c.get("source", "")
        hits = [name for name in _markers if name in s]
        if hits:
            print(f"  cell {i:2d}: {', '.join(hits)}")
else:
    print("Phase-1 notebook not found at", PHASE1_NB_PATH, "(continuing with defaults)")

# -----------------------------------------------------------------------------
# Discover optional Phase-1 artifacts that we can reuse.
# -----------------------------------------------------------------------------
WANTED_P1_FILES = [
    "document_embeddings_4B.npz",
    "dev_retrieval_top10.jsonl",
    "hidden_retrieval_top10.jsonl",
    "NLP_phase1_output/dev/dense/predictions.jsonl",
    "NLP_phase1_output/dev/dense/run_config.yaml",
    "NLP_phase1_output/hidden/dense/predictions.jsonl",
]

_roots = []
if CONFIG["phase1_outputs_dir"]:
    _roots.append(Path(CONFIG["phase1_outputs_dir"]))
_roots.append(DATASET_ROOT)

_roots = [r for r in _roots if r.exists()]


def _find_phase1_artifact(rel, roots):
    """Look for a Phase-1 artifact at <root>/rel, <root>/NLP_phase1/rel, or via rglob.

    Phase-2 outputs under OUTPUT_DIR are explicitly excluded so we don't
    accidentally use our own files as "Phase-1" artifacts.
    """
    rel_p = Path(rel)
    direct, rglob_hits = [], []
    for root in roots:
        for cand in (root / rel_p, root / "NLP_phase1" / rel_p):
            if cand.is_file() and OUTPUT_DIR not in cand.parents and "NLP_phase2" not in str(cand):
                direct.append(cand)
        try:
            for hit in root.rglob(rel_p.name):
                if not hit.is_file():
                    continue
                if OUTPUT_DIR in hit.parents or "NLP_phase2" in str(hit):
                    continue
                if not str(hit).endswith(rel):
                    continue
                rglob_hits.append(hit)
        except OSError:
            pass
    if direct:
        return sorted(direct, key=lambda p: ("phase1" not in str(p).lower(), str(p)))[0]
    if rglob_hits:
        return sorted(rglob_hits, key=lambda p: ("phase1" not in str(p).lower(), str(p)))[0]
    return None


PHASE1_ARTIFACTS = {rel: _find_phase1_artifact(rel, _roots) for rel in WANTED_P1_FILES}

print("\nPhase-1 artifacts:")
for rel, path in PHASE1_ARTIFACTS.items():
    status = f"OK  {path}" if path else "--  (not found)"
    print(f"  {status}  {rel}")

# Phase-1 predictions are mandatory; passage embeddings may be generated once.
for _required in ("NLP_phase1_output/dev/dense/predictions.jsonl", "NLP_phase1_output/dev/dense/run_config.yaml"):
    if PHASE1_ARTIFACTS[_required] is None:
        raise FileNotFoundError(f"Required Phase-1 artifact not found: {_required}.")
if PHASE1_ARTIFACTS["document_embeddings_4B.npz"] is None:
    print("Phase-1 passage embeddings not found in the dataset; a compatible matrix will be generated or reused from the Phase-2 working cache.")


Phase-1 notebook not found at /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/nlp-phase1.ipynb (continuing with defaults)

Phase-1 artifacts:
  --  (not found)  document_embeddings_4B.npz
  --  (not found)  dev_retrieval_top10.jsonl
  --  (not found)  hidden_retrieval_top10.jsonl
  OK  /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/NLP_phase1_output/dev/dense/predictions.jsonl  NLP_phase1_output/dev/dense/predictions.jsonl
  OK  /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/NLP_phase1_output/dev/dense/run_config.yaml  NLP_phase1_output/dev/dense/run_config.yaml
  OK  /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/NLP_phase1_output/hidden/dense/predictions.jsonl  NLP_phase1_output/hidden/dense/predictions.jsonl
Phase-1 passage embeddings not found in the dataset; a compatible matrix will be generated or reused from the Phase-2 working cache.


## §2 Passage, query and sentence embeddings

The Phase-1 document embedding file is reused when present. Otherwise, document vectors are generated using the original Phase-1 `make_embeddings`, saved as `artifacts/document_embeddings_4B.npz`, and reused on later runs when the metadata matches. Query and sentence embeddings are cached separately.


In [5]:
# =============================================================================
# §2  Passage, query and sentence embeddings
# =============================================================================
PHASE1_UTILS = globals().get("PHASE1_UTILS")

# Query prefix comes from Phase 1 (QUERY_INSTRUCT is exported by §1 from
# the supplied Phase-1 Python export).
print("query prefix:", repr(QUERY_INSTRUCT))


def resolve_max_memory():
    """Build an int-keyed max_memory dict containing only GPUs that are actually visible.

    Newer versions of accelerate reject string keys like "0"/"1" with
    "Device 0 is not recognized", and also reject keys for GPUs that do not
    exist (e.g. key 1 when only one GPU is visible).
    """
    n = torch.cuda.device_count()
    if n == 0:
        return None                       # let device_map="auto" decide
    template = CONFIG.get("gpu_memory", {})
    caps = {}
    for i in range(n):
        caps[i] = template.get(i, template.get(str(i), "13GiB"))
    caps["cpu"] = template.get("cpu", "40GiB")
    print("max_memory (int keys):", caps)
    return caps


def load_causal(cls, name, **kw):
    """Robust wrapper around `cls.from_pretrained`.

    Tries, in order:
        1. the supplied kwargs,
        2. with torch_dtype renamed to dtype (newer transformers),
        3. additionally with max_memory removed (fallback for strict accelerate).
    Also enables low_cpu_mem_usage and tries the memory-efficient "sdpa"
    attention implementation when available.
    """
    kw.setdefault("low_cpu_mem_usage", True)
    torch.cuda.empty_cache()
    gc.collect()

    variants = [dict(kw)]

    sdpa_kw = dict(kw)
    sdpa_kw.setdefault("attn_implementation", "sdpa")
    if sdpa_kw != variants[0]:
        variants.append(sdpa_kw)

    renamed = dict(sdpa_kw)
    if "torch_dtype" in renamed:
        renamed["dtype"] = renamed.pop("torch_dtype")
    if renamed != variants[-1]:
        variants.append(renamed)

    if "max_memory" in renamed:
        no_mm = dict(renamed)
        no_mm.pop("max_memory", None)
        variants.append(no_mm)

    last_exc = None
    for i, variant in enumerate(variants):
        try:
            return cls.from_pretrained(name, **variant)
        except (TypeError, ValueError) as exc:
            last_exc = exc
            msg = str(exc)
            relevant = any(t in msg for t in
                           ("torch_dtype", "dtype", "max_memory", "Device", "not recognized",
                            "attn_implementation"))
            if i < len(variants) - 1 and relevant:
                print(f"WARNING: retry {i + 1}/{len(variants)} for {name}: {msg[:170]}")
                continue
            raise
    raise last_exc


def free_gpu(*objs):
    """Delete references, run GC and empty the CUDA cache."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()


def load_embedder():
    tok = AutoTokenizer.from_pretrained(CONFIG["embedding_model"], padding_side="left")
    mdl = load_causal(
        AutoModel, CONFIG["embedding_model"],
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=resolve_max_memory(),
    )
    mdl.eval()
    return tok, mdl


# Use the Phase-1 embedding function with a Phase-2 interface adapter.
# make_embeddings is wrapped only to keep the Phase-2 progress logging
# (the core math is exactly Phase 1's make_embeddings).
query_texts_for = PHASE1_UTILS.query_texts_for


def make_embeddings(texts, tokenizer, model, batch_size=4, max_length=1024, desc=""):
    """Phase-1 embedding recipe with Phase-2 progress logging."""
    t0 = time.time()
    out = PHASE1_UTILS.make_embeddings(
        texts, tokenizer, model, batch_size=batch_size, max_length=max_length)
    if desc:
        print(f"  {desc}: {len(texts)} texts in {time.time() - t0:.0f}s", flush=True)
    return out


def align_embeddings(ids, emb, target_ids):
    """Reorder embedding rows to match target_ids and assert full coverage."""
    idx = {d: i for i, d in enumerate(ids)}
    missing = [d for d in target_ids if d not in idx]
    assert not missing, f"{len(missing)} doc ids missing in cached embeddings, e.g. {missing[:3]}"
    order = np.array([idx[d] for d in target_ids])
    return emb[order]


# ------------------------------------------------------------------ passages
import hashlib

# Cache metadata binds generated embeddings to corpus contents and Phase-1 code.
PASSAGE_EMB_PATH = ARTIFACT_DIR / "document_embeddings_4B.npz"
PASSAGE_META_PATH = ARTIFACT_DIR / "document_embeddings_4B.meta.json"
_hash = hashlib.sha256()
for _id, _text in zip(DOC_IDS, PASSAGE_TEXTS):
    _hash.update(json.dumps([_id, _text], ensure_ascii=False).encode("utf-8"))
    _hash.update(b"\n")
_current_meta = {
    "corpus_sha256": _hash.hexdigest(),
    "phase1_source_sha256": hashlib.sha256(_p1_source.encode("utf-8")).hexdigest(),
    "embedding_model": CONFIG["embedding_model"],
    "max_length": CONFIG["embed_max_len"],
    "pooling": "last_token_l2_normalized",
}

def _read_passage_embeddings(path):
    with np.load(path, allow_pickle=False) as z:
        ids = [str(x) for x in z["doc_ids"]]
        emb = z["embeddings"]
        if (emb.ndim != 2 or emb.shape[0] != len(ids) or len(ids) != len(set(ids))
                or set(ids) != set(DOC_IDS) or not np.isfinite(emb).all()):
            raise ValueError("Embedding matrix or document IDs do not match the current corpus.")
        norms = np.linalg.norm(emb.astype(np.float32), axis=1)
        if not np.allclose(norms, 1.0, atol=1e-3):
            raise ValueError("Passage embeddings must be L2-normalized.")
        return align_embeddings(ids, emb, DOC_IDS).astype(np.float32, copy=False)

_p1_embedding_path = PHASE1_ARTIFACTS.get("document_embeddings_4B.npz")
passage_emb = None
if _p1_embedding_path is not None:
    try:
        passage_emb = _read_passage_embeddings(_p1_embedding_path)
    except Exception as exc:
        raise RuntimeError(f"Cannot use Phase-1 embeddings {_p1_embedding_path}: {exc}") from exc
    PASSAGE_EMB_ORIGIN = f"Phase-1 dataset file: {_p1_embedding_path}"
    print("loaded Phase-1 passage embeddings:", _p1_embedding_path)
    print("WARNING: an external embedding file cannot by itself prove model or corpus provenance.")
elif PASSAGE_EMB_PATH.exists() and PASSAGE_META_PATH.exists():
    try:
        if json.loads(PASSAGE_META_PATH.read_text(encoding="utf-8")) == _current_meta:
            passage_emb = _read_passage_embeddings(PASSAGE_EMB_PATH)
            PASSAGE_EMB_ORIGIN = f"Generated Phase-2 cache: {PASSAGE_EMB_PATH}"
            print("reused verified passage embeddings:", PASSAGE_EMB_PATH)
        else:
            print("Passage cache metadata differs; regenerating the matrix.")
    except Exception as exc:
        print(f"Passage cache invalid; regenerating: {exc}")

need_embedder = passage_emb is None
_query_meta_path = ARTIFACT_DIR / "query_embeddings.meta.json"
_sent_meta_path = ARTIFACT_DIR / "sentence_embeddings.meta.json"
_query_meta = dict(_current_meta, query_sha256=hashlib.sha256(
    json.dumps([questions_dev, questions_hidden, QUERY_INSTRUCT], ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest())
_sent_meta = dict(_current_meta, sentence_max_length=CONFIG["sent_max_len"],
                  sentence_sha256=hashlib.sha256(json.dumps(DOC_SENTS, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest())
def _valid_metadata(meta_path, expected):
    try:
        return json.loads(meta_path.read_text(encoding="utf-8")) == expected
    except (OSError, ValueError):
        return False
need_sentences = CONFIG["run_sentence"] and not (
    (ARTIFACT_DIR / "sentence_embeddings.npz").is_file() and _valid_metadata(_sent_meta_path, _sent_meta))
have_query_cache = ((ARTIFACT_DIR / "query_embeddings.npz").is_file()
                    and _valid_metadata(_query_meta_path, _query_meta))

if need_embedder or need_sentences or not have_query_cache:
    print("\nloading embedding model:", CONFIG["embedding_model"])
    emb_tok, emb_model = load_embedder()

    if need_embedder:
        print("Creating document embeddings with the original Phase-1 make_embeddings function.")
        _batch = max(1, int(CONFIG["embed_batch"]))
        while True:
            try:
                passage_emb = make_embeddings(PASSAGE_TEXTS, emb_tok, emb_model,
                                              batch_size=_batch, max_length=CONFIG["embed_max_len"],
                                              desc="passages")
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                gc.collect()
                if _batch == 1:
                    raise
                _batch = max(1, _batch // 2)
                print(f"Passage embedding OOM: retrying with batch_size={_batch}")
        if passage_emb.ndim != 2 or passage_emb.shape[0] != len(DOC_IDS) or not np.isfinite(passage_emb).all():
            raise ValueError("Generated passage embeddings are invalid.")
        _tmp = PASSAGE_EMB_PATH.with_name("document_embeddings_4B.tmp.npz")
        np.savez_compressed(_tmp, doc_ids=np.asarray(DOC_IDS, dtype=str),
                            embeddings=passage_emb.astype(np.float32))
        _tmp.replace(PASSAGE_EMB_PATH)
        PASSAGE_META_PATH.write_text(json.dumps(_current_meta, indent=2), encoding="utf-8")
        passage_emb = _read_passage_embeddings(PASSAGE_EMB_PATH)
        PASSAGE_EMB_ORIGIN = f"Generated from Phase-1 function: {PASSAGE_EMB_PATH}"
        print("Saved passage embeddings:", PASSAGE_EMB_PATH)
        print("Save the Kaggle output or upload this NPZ to the input dataset for future sessions.")

    # Reuse the query cache whenever it is present.
    if not have_query_cache:
        dev_q_emb = make_embeddings(query_texts_for(questions_dev), emb_tok, emb_model,
                                    batch_size=CONFIG["embed_batch"],
                                    max_length=CONFIG["embed_max_len"], desc="dev queries")
        hid_q_emb = make_embeddings(query_texts_for(questions_hidden), emb_tok, emb_model,
                                    batch_size=CONFIG["embed_batch"],
                                    max_length=CONFIG["embed_max_len"], desc="hidden queries")
        np.savez_compressed(ARTIFACT_DIR / "query_embeddings.npz",
                            dev_qids=np.asarray([str(q["qid"]) for q in questions_dev]),
                            dev=dev_q_emb,
                            hidden_qids=np.asarray([str(q["qid"]) for q in questions_hidden]),
                            hidden=hid_q_emb)
        _query_meta_path.write_text(json.dumps(_query_meta, indent=2), encoding="utf-8")
    
    # sentences
    if need_sentences:
        print("computing sentence embeddings ...")
        sent_rows, sent_texts = [], []
        for d in DOC_IDS:
            for si, s in enumerate(DOC_SENTS.get(d, [])):
                if s and s.strip():
                    sent_rows.append((d, si))
                    sent_texts.append(f"Title: {DOC_TITLE.get(d, '')}\nSentence: {s.strip()}")
        sent_emb = make_embeddings(sent_texts, emb_tok, emb_model,
                                   batch_size=CONFIG["sent_batch"],
                                   max_length=CONFIG["sent_max_len"], desc="sentences")
        np.savez_compressed(ARTIFACT_DIR / "sentence_embeddings.npz",
                            doc_ids=np.asarray([r[0] for r in sent_rows]),
                            sent_idx=np.asarray([r[1] for r in sent_rows], dtype=np.int32),
                            embeddings=sent_emb)
        _sent_meta_path.write_text(json.dumps(_sent_meta, indent=2), encoding="utf-8")
        print("sentence embeddings:", sent_emb.shape)

    # The imported Phase-1 function retains tokenizer/model in its globals.
    _p1_globals.pop("tokenizer", None)
    _p1_globals.pop("model", None)
    del emb_tok, emb_model
    gc.collect()
    torch.cuda.empty_cache()
    print("embedding model unloaded")
else:
    print("all embeddings are cached; embedding model not needed")

# ------------------------------------------------------------------ load from cache
PASSAGE_EMB = passage_emb
assert PASSAGE_EMB.shape[0] == len(DOC_IDS)
assert np.isfinite(PASSAGE_EMB).all()

_z = np.load(ARTIFACT_DIR / "query_embeddings.npz", allow_pickle=False)
assert [str(x) for x in _z["dev_qids"]] == [str(q["qid"]) for q in questions_dev]
assert [str(x) for x in _z["hidden_qids"]] == [str(q["qid"]) for q in questions_hidden]
DEV_Q_EMB = _z["dev"].astype(np.float32)
HID_Q_EMB = _z["hidden"].astype(np.float32)
if (DEV_Q_EMB.ndim != 2 or HID_Q_EMB.ndim != 2 or
        DEV_Q_EMB.shape[1] != PASSAGE_EMB.shape[1] or HID_Q_EMB.shape[1] != PASSAGE_EMB.shape[1] or
        not np.isfinite(DEV_Q_EMB).all() or not np.isfinite(HID_Q_EMB).all()):
    raise ValueError("Query embeddings are invalid or incompatible with passage embeddings.")

if CONFIG["run_sentence"]:
    _s = np.load(ARTIFACT_DIR / "sentence_embeddings.npz", allow_pickle=False)
    SENT_DOC_IDS = [str(x) for x in _s["doc_ids"]]
    SENT_IDX     = _s["sent_idx"].astype(np.int32)
    SENT_EMB     = _s["embeddings"].astype(np.float32)
    SENT_DOC_ROW = np.array([DOC_IDX[d] for d in SENT_DOC_IDS], dtype=np.int32)
    print(f"sentence matrix: {SENT_EMB.shape} ({len(SENT_DOC_IDS)} sentences)")

print("passage embeddings:", passage_emb.shape,
      "| dev queries:", DEV_Q_EMB.shape, "| hidden queries:", HID_Q_EMB.shape)


query prefix: 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '

loading embedding model: Qwen/Qwen3-Embedding-4B


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Creating document embeddings with the original Phase-1 make_embeddings function.
  passages: 3882 texts in 418s
Saved passage embeddings: /kaggle/working/NLP_phase2/artifacts/document_embeddings_4B.npz
Save the Kaggle output or upload this NPZ to the input dataset for future sessions.
  dev queries: 300 texts in 6s
  hidden queries: 300 texts in 6s
computing sentence embeddings ...
  sentences: 14048 texts in 489s
sentence embeddings: (14048, 2560)
embedding model unloaded
sentence matrix: (14048, 2560) (14048 sentences)
passage embeddings: (3882, 2560) | dev queries: (300, 2560) | hidden queries: (300, 2560)


## §3 Hand-written BM25

Implements the standard BM25 scoring function:

\[
\text{score}(q,d)=\sum_{t\in q}\text{idf}(t)\cdot\frac{f_{td}\cdot(k_1+1)}{f_{td}+k_1\cdot(1-b+b\cdot|d|/\text{avgdl})}
\]

with \(k_1=1.5\), \(b=0.75\), and positive IDF defined as

\[
\text{idf}(t)=\ln\!\left(1+\frac{N-df_t+0.5}{df_t+0.5}\right).
\]

Properties:

- OOV terms are silently skipped (no crash on out-of-vocabulary words).
- Document masks (`exclude_ids`) are applied **before** `top-k`, so masked documents never appear in the returned list even when `k` is large relative to the remaining pool.
- Deterministic tie-break by ascending `doc_id`.


In [6]:
# =============================================================================
# §3  Hand-written BM25
# =============================================================================
TOKEN_RE = re.compile(r"[a-z0-9']+")

# A small, transparent English stop-word list.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "he", "in", "is", "it", "its", "of", "on", "that", "the",
    "to", "was", "were", "will", "with",
}


def tokenize(text, remove_stopwords=True):
    toks = TOKEN_RE.findall(str(text).lower())
    if remove_stopwords:
        toks = [t for t in toks if t not in STOPWORDS]
    return toks


class BM25Index:
    def __init__(self, doc_ids, texts, k1=1.5, b=0.75, remove_stopwords=True):
        self.doc_ids = [str(d) for d in doc_ids]
        self.k1 = float(k1)
        self.b = float(b)
        self.remove_stopwords = bool(remove_stopwords)
        self.N = len(self.doc_ids)

        tokens_per_doc = [tokenize(t, remove_stopwords) for t in texts]
        self.doc_len = np.array([len(toks) for toks in tokens_per_doc], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N > 0 else 1.0

        # postings: term -> {doc_row: tf}
        df = Counter()
        postings_raw = defaultdict(dict)
        for i, toks in enumerate(tokens_per_doc):
            tf = Counter(toks)
            for term, count in tf.items():
                postings_raw[term][i] = count
            for term in tf.keys():
                df[term] += 1

        # positive IDF (BM25+ style variant used in the handout)
        self.idf = {term: math.log(1.0 + (self.N - c + 0.5) / (c + 0.5))
                    for term, c in df.items()}

        # Pre-compact postings into numpy arrays for fast scoring.
        self.postings = {}
        for term, row_tf in postings_raw.items():
            rows = np.array(sorted(row_tf.keys()), dtype=np.int64)
            tfs = np.array([row_tf[r] for r in rows], dtype=np.float32)
            self.postings[term] = (rows, tfs)

        # per-document normalization factor: k1 * (1 - b + b * |d| / avgdl)
        self.norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / max(self.avgdl, 1e-9))

    def score(self, query):
        """Return an (N,) array of BM25 scores for all documents."""
        scores = np.zeros(self.N, dtype=np.float32)
        for term in tokenize(query, self.remove_stopwords):
            if term not in self.postings:
                continue                           # OOV term -> safely ignored
            rows, tfs = self.postings[term]
            idf = self.idf[term]
            # Standard BM25 term contribution.
            scores[rows] += idf * (tfs * (self.k1 + 1.0)) / (tfs + self.norm[rows])
        return scores

    def search(self, query, k=10, exclude_ids=frozenset()):
        """Return the top-k doc ids, with masked documents removed BEFORE top-k."""
        if k <= 0:
            raise ValueError("k must be positive")
        s = self.score(query)
        excluded = {str(x) for x in exclude_ids}
        excl = {i for i, d in enumerate(self.doc_ids) if str(d) in excluded}
        if excl:
            s = s.copy()
            s[list(excl)] = -np.inf
        k_eff = max(1, min(k, self.N - len(excl)))
        # Stable argsort on (-score, doc_id) for deterministic tie-breaking.
        order = np.lexsort((np.array(self.doc_ids), -s))
        return [self.doc_ids[i] for i in order[:k_eff] if np.isfinite(s[i])]

    def explain(self, query, doc_id):
        """Return per-term contributions for one document (used in error analysis)."""
        row = self.doc_ids.index(str(doc_id))
        out = {}
        for term in tokenize(query, self.remove_stopwords):
            if term not in self.postings:
                out[term] = 0.0
                continue
            rows, tfs = self.postings[term]
            if row not in set(int(r) for r in rows):
                out[term] = 0.0
                continue
            j = list(int(r) for r in rows).index(row)
            tf = float(tfs[j])
            out[term] = float(self.idf[term] * tf * (self.k1 + 1.0) / (tf + self.norm[row]))
        return out


BM25 = BM25Index(DOC_IDS, PASSAGE_TEXTS,
                 k1=CONFIG["bm25_k1"], b=CONFIG["bm25_b"],
                 remove_stopwords=CONFIG["bm25_remove_stopwords"])

# -------------- self-tests
_top = BM25.search(questions_dev[0]["question"], k=10)
assert len(_top) == 10 and len(set(_top)) == 10
_excl = {_top[0]}
assert _top[0] not in BM25.search(questions_dev[0]["question"], k=10, exclude_ids=_excl)
# mask should never produce fewer than k results when enough unmasked docs exist
_mask_n = max(1, min(5, BM25.N - 10))
assert len(BM25.search(questions_dev[0]["question"], k=10,
                       exclude_ids=set(DOC_IDS[:_mask_n]))) == 10
# OOV query must not crash and should return all-zero scores effectively
_oov = BM25.search("zzzznonexistentword xyzqzz", k=5)
assert isinstance(_oov, list)
# empty query
assert isinstance(BM25.search("", k=5), list)
# k larger than N
assert len(BM25.search(questions_dev[0]["question"], k=BM25.N + 100)) <= BM25.N
# finite scores
_q = BM25.score(questions_dev[0]["question"])
assert np.isfinite(_q).all()
assert np.isfinite(list(BM25.idf.values())).all()
# mask everything
assert BM25.search(questions_dev[0]["question"], k=10, exclude_ids=set(DOC_IDS)) == []
print("BM25 self-checks passed. sample top-3:", _top[:3])

# Independent rank_bm25 validation; its default BM25Okapi IDF differs from the
# positive-IDF formula required by the handout, so align only the IDF variant.
try:
    from rank_bm25 import BM25Okapi
except ImportError as exc:
    raise ImportError("rank_bm25 is required for the reference self-check. Enable internet and install rank_bm25.") from exc
_reference_tokens = [tokenize(t, CONFIG["bm25_remove_stopwords"]) for t in PASSAGE_TEXTS]
_reference = BM25Okapi(_reference_tokens, k1=CONFIG["bm25_k1"], b=CONFIG["bm25_b"])
_raw_idf_reference = dict(_reference.idf)
_ref_df = Counter(term for doc in _reference_tokens for term in set(doc))
_reference.idf = {term: math.log1p((len(_reference_tokens) - df + 0.5) / (df + 0.5))
                  for term, df in _ref_df.items()}
_validation_queries = [q["question"] for q in questions_dev[:min(50, len(questions_dev))]]
_validation_queries += ["", "zzzznonexistentword xyzqzz", "film film director"]
_max_abs = 0.0
for _query in _validation_queries:
    _got = BM25.score(_query)
    _expected = np.asarray(_reference.get_scores(tokenize(_query, CONFIG["bm25_remove_stopwords"])), dtype=np.float32)
    _diff = float(np.max(np.abs(_got - _expected))) if len(_got) else 0.0
    _max_abs = max(_max_abs, _diff)
    if not np.allclose(_got, _expected, rtol=1e-5, atol=1e-5):
        raise AssertionError(f"BM25 score mismatch for query {_query!r}: max abs difference {_diff}")
    # Compare scores for a masked set as well; ranking of very close values is
    # separately affected by float32 ties and the deterministic ID tie-break.
    _masked = set(DOC_IDS[:min(5, max(0, len(DOC_IDS) - 10))])
    assert not (set(BM25.search(_query, k=min(10, len(DOC_IDS)), exclude_ids=_masked)) & _masked)
_bm25_check = {
    "library": "rank_bm25.BM25Okapi", "queries_tested": len(_validation_queries),
    "max_abs_score_diff": _max_abs, "reference_idf": "positive handout log1p variant",
    "rank_bm25_default_idf_note": "The library's default IDF differs; the reference IDF was aligned to the handout formula.",
    "default_idf_different_terms": sum(not np.isclose(_raw_idf_reference[t], _reference.idf[t]) for t in _reference.idf),
    "vocabulary_size": len(_reference.idf), "score_match": True,
}
(ARTIFACT_DIR / "bm25_reference_check.json").write_text(json.dumps(_bm25_check, indent=2), encoding="utf-8")
print("BM25 vs rank_bm25 reference check:", _bm25_check)


BM25 self-checks passed. sample top-3: ['p02917', 'p03320', 'p02916']
BM25 vs rank_bm25 reference check: {'library': 'rank_bm25.BM25Okapi', 'queries_tested': 53, 'max_abs_score_diff': 3.814697265625e-06, 'reference_idf': 'positive handout log1p variant', 'rank_bm25_default_idf_note': "The library's default IDF differs; the reference IDF was aligned to the handout formula.", 'default_idf_different_terms': 28034, 'vocabulary_size': 28034, 'score_match': True}


## §4 Reciprocal Rank Fusion

Fuses multiple ranked lists into a single ranking using

\[
\text{RRF}(d)=\sum_{r}\frac{1}{k+\text{rank}_r(d)}, \quad k=60,
\]

where `rank` starts at **1** (not 0). Documents that do not appear in a particular list contribute nothing from that source. Ties are broken deterministically by ascending `doc_id`. Masked documents (`exclude_ids`) are dropped before aggregation.


In [7]:
# =============================================================================
# §4  Reciprocal Rank Fusion
# =============================================================================
def rrf_fuse(rank_lists, rrf_k=60, top_k=10, exclude_ids=frozenset()):
    """Fuse multiple ranked lists via RRF:  score(d) = Σ 1/(rrf_k + rank_r(d)).

    rank starts at 1. Documents not present in a list receive zero contribution
    from that source. Tie-break is deterministic (ascending doc id).
    """
    assert rrf_k >= 0, "rrf_k must be >= 0"
    excl = set(str(x) for x in exclude_ids)
    scores = defaultdict(float)
    for lst in rank_lists:
        for rank, d in enumerate(lst, start=1):
            d = str(d)
            if d in excl:
                continue
            scores[d] += 1.0 / (rrf_k + rank)
    ordered = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    return [d for d, _ in ordered[:top_k]]


def rrf_scores(rank_lists, rrf_k=60, exclude_ids=frozenset()):
    """Return the full RRF score dict (for reporting/analysis)."""
    excl = set(str(x) for x in exclude_ids)
    scores = defaultdict(float)
    for lst in rank_lists:
        for rank, d in enumerate(lst, start=1):
            d = str(d)
            if d in excl:
                continue
            scores[d] += 1.0 / (rrf_k + rank)
    return dict(scores)


# -------------- self-tests
_a = ["p001", "p002", "p003", "p004"]
_b = ["p002", "p001", "p004", "p003"]
# Tie between p001 and p002 (both appear once at rank 1 and once at rank 2) -> smaller id wins
assert rrf_fuse([_a, _b], rrf_k=60, top_k=4) == ["p001", "p002", "p003", "p004"] or \
       rrf_fuse([_a, _b], rrf_k=60, top_k=4)[:2] == ["p001", "p002"]
assert len(rrf_fuse([_a, _b], rrf_k=60, top_k=4)) == 4
# rank 1 -> 1/61
_pts = rrf_scores([["pA"]], rrf_k=60)
assert abs(_pts["pA"] - 1.0 / 61.0) < 1e-9
# rrf_k=0 -> harmonic-style 1/rank
_pts0 = rrf_scores([["pA", "pB"], ["pA"]], rrf_k=0)
assert abs(_pts0["pA"] - (1.0 + 1.0)) < 1e-9
# pB only in list 2 at rank 3 -> 1/63
_mix = rrf_scores([["pA", "pB"], ["pA", "pC", "pB"]], rrf_k=60)
assert abs(_mix["pB"] - (1.0 / 62.0 + 1.0 / 63.0)) < 1e-9
# excluded id must not appear
assert "pA" not in rrf_fuse([_a, _b], rrf_k=60, top_k=4, exclude_ids={"pA"})
# deterministic: same input -> same output across two calls
assert rrf_fuse([_a, _b]) == rrf_fuse([_a, _b])
print("RRF self-checks passed")


RRF self-checks passed


## §5 Dense and sentence-level retrieval

Because the corpus is small (3,882 passages), dense retrieval uses an **exact** full-matrix dot product between the query vector and all passage vectors; since both are L2-normalized this equals cosine similarity.

For the sentence-level experiment:

1. Every sentence in every passage is embedded individually (with a small `Title:` + `Sentence:` prefix).
2. Sentence similarities are **max-pooled** per passage (`np.maximum.at`), so each passage receives the score of its best-matching sentence.
3. Retrieval still returns passage IDs, so downstream context packing / reranking / LLM code stays unchanged.


In [8]:
# =============================================================================
# §5  Exact dense search and sentence-level max-pool retrieval
# =============================================================================
# Adapt the original Phase-1 dense_search to Phase-2 score arrays.
topk_from_scores = dense_search_from_scores


def dense_scores(q_vec, exclude_ids=frozenset()):
    """Cosine similarity of a query vector against all passages.

    Since both matrices are L2-normalized, the dot product equals cosine.
    """
    sim = PASSAGE_EMB @ np.asarray(q_vec, dtype=np.float32)
    return sim


def dense_search(q_vec, k=10, exclude_ids=frozenset()):
    return topk_from_scores(dense_scores(q_vec, exclude_ids), DOC_IDS, k=k, exclude_ids=exclude_ids)


def sentence_scores(q_vec, exclude_ids=frozenset()):
    """Per-passage score defined as the MAX similarity over its sentences."""
    sims = SENT_EMB @ np.asarray(q_vec, dtype=np.float32)
    pass_scores = np.full(len(DOC_IDS), -np.inf, dtype=np.float32)
    np.maximum.at(pass_scores, SENT_DOC_ROW, sims)
    excl = {DOC_IDX[d] for d in exclude_ids if d in DOC_IDX}
    if excl:
        pass_scores[list(excl)] = -np.inf
    return pass_scores


def sentence_search(q_vec, k=10, exclude_ids=frozenset()):
    return topk_from_scores(sentence_scores(q_vec, exclude_ids), DOC_IDS, k=k, exclude_ids=exclude_ids)


def best_sentence(doc_id, q_vec):
    """Return the best-scoring sentence text for (doc, query) — used in error analysis."""
    doc_id = str(doc_id)
    sents = DOC_SENTS.get(doc_id, [])
    if not sents:
        return ""
    mask = np.array([i for i, d in enumerate(SENT_DOC_IDS) if d == doc_id], dtype=np.int64)
    if mask.size == 0:
        return ""
    sims = SENT_EMB[mask] @ np.asarray(q_vec, dtype=np.float32)
    return sents[int(np.argmax(sims))]


# -------------- self-tests
_q0 = DEV_Q_EMB[0]
_dtop = dense_search(_q0, k=10)
assert len(_dtop) == 10 and len(set(_dtop)) == 10
assert _dtop[0] in DOC_IDX
_excl = {_dtop[0]}
assert _dtop[0] not in dense_search(_q0, k=10, exclude_ids=_excl)
assert dense_scores(_q0).max() == dense_scores(_q0)[DOC_IDX[_dtop[0]]]
if CONFIG["run_sentence"]:
    _s = sentence_search(_q0, k=10)
    assert len(_s) == 10 and len(set(_s)) == 10
    assert isinstance(best_sentence(_s[0], _q0), str)
print("Dense search self-checks passed. dense top-3:", _dtop[:3])


Dense search self-checks passed. dense top-3: ['p02917', 'p02916', 'p03365']


## §6 Two optional rerankers

BGE and Qwen3-Reranker-0.6B separately rescore the same RRF candidate IDs. BGE uses classifier logits; Qwen uses its official causal yes/no scoring template. They are loaded sequentially and unloaded before answer generation. Failures are explicit: no silent model substitution.


In [9]:
# =============================================================================
# §6  Optional BGE and Qwen3 rerankers, loaded sequentially
# =============================================================================
from transformers import AutoModelForSequenceClassification

RERANKER = None
RERANK_TIMINGS = {}
QWEN_RERANK_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
QWEN_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
QWEN_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


def load_reranker(kind):
    global RERANKER
    unload_reranker()
    t0 = time.perf_counter()
    name = CONFIG['bge_reranker_model'] if kind == 'bge' else CONFIG['qwen_reranker_model']
    if kind not in ('bge', 'qwen'):
        raise ValueError(f'Unknown reranker: {kind}')
    tok = AutoTokenizer.from_pretrained(name, padding_side='left' if kind == 'qwen' else 'right')
    cls = AutoModelForSequenceClassification if kind == 'bge' else AutoModelForCausalLM
    model = load_causal(cls, name, torch_dtype=torch.float16)
    model.eval()
    model.to('cuda:0')
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if kind == 'qwen':
        yes = tok.convert_tokens_to_ids('yes')
        no = tok.convert_tokens_to_ids('no')
        if yes is None or no is None or yes == no or yes == tok.unk_token_id or no == tok.unk_token_id:
            raise ValueError('Qwen yes/no tokens are not valid single tokens')
        prefix = tok.encode(QWEN_PREFIX, add_special_tokens=False)
        suffix = tok.encode(QWEN_SUFFIX, add_special_tokens=False)
        if len(prefix) + len(suffix) >= CONFIG['rerank_max_length']:
            raise ValueError('rerank_max_length is too small for the Qwen template')
        RERANKER = (kind, tok, model, (yes, no, prefix, suffix))
    else:
        RERANKER = (kind, tok, model, None)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    RERANK_TIMINGS[kind] = {'model': name, 'load_seconds': time.perf_counter()-t0,
                            'scoring_seconds': 0.0, 'pairs': 0}
    print('reranker loaded:', kind, name)


def unload_reranker():
    global RERANKER
    if RERANKER is not None:
        RERANKER = None
        gc.collect()
        torch.cuda.empty_cache()


@torch.inference_mode()
def _score_rerank_batch(query, texts):
    kind, tok, model, extra = RERANKER
    device = model.get_input_embeddings().weight.device
    if kind == 'bge':
        enc = tok([[query, t] for t in texts], padding=True, truncation=True,
                  max_length=CONFIG['rerank_max_length'], return_tensors='pt')
    else:
        yes, no, prefix, suffix = extra
        pairs = [f'<Instruct>: {QWEN_RERANK_INSTRUCTION}\n<Query>: {query}\n<Document>: {t}' for t in texts]
        encoded = tok(pairs, padding=False, truncation=True, add_special_tokens=False,
                      max_length=CONFIG['rerank_max_length'] - len(prefix) - len(suffix))
        enc = tok.pad({'input_ids': [prefix + ids + suffix for ids in encoded['input_ids']]},
                      padding=True, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    if kind == 'qwen':
        # Difference of yes/no logits has the same order as their two-way softmax.
        logits = model(**enc, use_cache=False).logits[:, -1, :]
        values = logits[:, yes].float() - logits[:, no].float()
    else:
        logits = model(**enc).logits.float()
        values = logits.squeeze(-1) if logits.size(-1) == 1 else logits[:, 1]
    return values.cpu().tolist()


def _score_with_oom_retry(query, texts):
    try:
        return _score_rerank_batch(query, texts)
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        if len(texts) <= 1:
            raise
        mid = len(texts) // 2
        print(f'Rerank OOM: retrying batch {len(texts)} as {mid}+{len(texts)-mid}')
        return _score_with_oom_retry(query, texts[:mid]) + _score_with_oom_retry(query, texts[mid:])


def rerank_search(query, candidate_ids, k=10):
    assert RERANKER is not None, 'reranker is not loaded'
    kind = RERANKER[0]
    t0 = time.perf_counter()
    bs = CONFIG['bge_rerank_batch'] if kind == 'bge' else CONFIG['qwen_rerank_batch']
    texts = [passage_text(d) for d in candidate_ids]
    scores = []
    for start in range(0, len(texts), bs):
        scores.extend(_score_with_oom_retry(query, texts[start:start + bs]))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    RERANK_TIMINGS[kind]['scoring_seconds'] += time.perf_counter() - t0
    RERANK_TIMINGS[kind]['pairs'] += len(texts)
    if len(scores) != len(candidate_ids) or not all(math.isfinite(x) for x in scores):
        raise ValueError(f'Invalid reranker scores: {kind}')
    ranked = sorted(zip(candidate_ids, scores), key=lambda x: (-x[1], x[0]))
    return [d for d, _ in ranked[:k]]


## §7 Building ranking tables

For every question (dev and hidden) we build ranked lists for the active systems:

| System | Description |
|---|---|
| `dense` | Pure dense retrieval baseline. |
| `bm25` | Pure BM25 baseline. |
| `hybrid` | RRF fusion of `dense` + `bm25`. |
| `hybrid_sentence` | RRF fusion of sentence-level dense + passage BM25. |
| `hybrid_bge` and `hybrid_qwen` | The same top 30 hybrid candidates rescored by BGE and Qwen separately. |
| `hybrid_goldmask` | Hybrid retrieval with gold documents **masked out** (abstention experiment). |

Each returned list always contains exactly 10 unique valid doc IDs, even when masks are applied.


In [10]:
# =============================================================================
# §7  Build ranking tables for dev and hidden splits
# =============================================================================
def gold_ids_of(qid):
    lab = LABELS_BY_QID.get(str(qid))
    return [str(x) for x in (lab["gold_doc_ids"] if lab else [])]


def build_rankings(items, q_emb, with_goldmask=False, tag=""):
    """For each question, build per-system rankings: dense/bm25/hybrid + (optional)
    sentence, rerank, and goldmask.

    Returns: dict {system: {qid: [doc_ids, ...]}}
    """
    systems = ["dense", "bm25", "hybrid"]
    if CONFIG["run_sentence"]:
        systems.append("hybrid_sentence")
    if with_goldmask and CONFIG["run_goldmask"]:
        systems.append("hybrid_goldmask")

    out = {s: {} for s in systems}
    t0 = time.time()

    for i, item in enumerate(items):
        qid, qtext, vec = str(item["qid"]), item["question"], q_emb[i]

        # Standard runs.
        d_list = dense_search(vec, CONFIG["fusion_depth"])
        b_list = BM25.search(qtext, CONFIG["fusion_depth"])
        h_list = rrf_fuse([d_list, b_list], CONFIG["rrf_k"], CONFIG["k_output"])

        out["dense"][qid] = d_list[:CONFIG["k_output"]]
        out["bm25"][qid] = b_list[:CONFIG["k_output"]]
        out["hybrid"][qid] = h_list

        if CONFIG["run_sentence"]:
            s_list = sentence_search(vec, CONFIG["fusion_depth"])
            out["hybrid_sentence"][qid] = rrf_fuse(
                [s_list, b_list], CONFIG["rrf_k"], CONFIG["k_output"])

        if CONFIG["run_rerank"]:
            RERANK_CANDIDATES[tag][qid] = rrf_fuse(
                [d_list, b_list], CONFIG["rrf_k"], CONFIG["rerank_candidates"])

        # Gold-mask abstention experiment: mask out all gold documents.
        if with_goldmask and CONFIG["run_goldmask"]:
            excl = frozenset(gold_ids_of(qid))
            d_m = dense_search(vec, CONFIG["fusion_depth"], excl)
            b_m = BM25.search(qtext, CONFIG["fusion_depth"], excl)
            out["hybrid_goldmask"][qid] = rrf_fuse(
                [d_m, b_m], CONFIG["rrf_k"], CONFIG["k_output"], excl)

        if (i + 1) % 50 == 0 or i + 1 == len(items):
            print(f"  [{tag}] {i + 1}/{len(items)} questions  "
                  f"({time.time() - t0:.0f}s)", flush=True)

    # Structural guarantees.
    for s, table in out.items():
        assert len(table) == len(items), f"{s}: {len(table)} rows for {len(items)} questions"
        for qid, ids in table.items():
            assert len(ids) == CONFIG["k_output"], f"{s}/{qid}: expected {CONFIG['k_output']} ids, got {len(ids)}"
            assert len(set(ids)) == len(ids), f"{s}/{qid}: duplicate ids"
            assert all(d in DOC_IDX for d in ids), f"{s}/{qid}: unknown doc id"
            if with_goldmask and s == "hybrid_goldmask":
                assert not (set(ids) & set(gold_ids_of(qid))), f"{qid}: gold doc leaked into goldmask output"

    return out


DEV_ITEMS = questions_dev if not CONFIG["dev_limit"] else questions_dev[:CONFIG["dev_limit"]]
DEV_EMB = DEV_Q_EMB if not CONFIG["dev_limit"] else DEV_Q_EMB[:CONFIG["dev_limit"]]

print(f"building rankings: dev={len(DEV_ITEMS)} questions, "
      f"hidden={len(questions_hidden)} questions")
RANKINGS = {}
RERANK_CANDIDATES = {"dev": {}, "hidden": {}}
if CONFIG["run_dev"]:
    RANKINGS["dev"] = build_rankings(DEV_ITEMS, DEV_EMB, with_goldmask=True, tag="dev")
    # Compare the regenerated dense rankings with the saved Phase-1 retrievals.
    # The controlled dense run uses the same regenerated vectors as hybrid.
    _p1_rows = read_jsonl(PHASE1_ARTIFACTS["NLP_phase1_output/dev/dense/predictions.jsonl"])
    _p1_by_qid = {str(r["qid"]): r for r in _p1_rows}
    if len(_p1_by_qid) != len(_p1_rows):
        raise ValueError("Duplicate qids in the saved Phase-1 dense predictions.")
    _pairs = []
    for _item in DEV_ITEMS:
        _qid = str(_item["qid"])
        if _qid not in _p1_by_qid:
            raise ValueError(f"Missing Phase-1 dense prediction for {_qid}")
        _old = [str(d) for d in _p1_by_qid[_qid]["retrieved"][:CONFIG["k_output"]]]
        _new = RANKINGS["dev"]["dense"][_qid]
        if len(_old) != CONFIG["k_output"] or len(set(_old)) != len(_old) or not set(_old) <= set(DOC_IDS):
            raise ValueError(f"Invalid Phase-1 dense ranking for {_qid}")
        _pairs.append({"qid": _qid, "same_order": _old == _new,
                       "same_set": set(_old) == set(_new),
                       "overlap_at_10": len(set(_old) & set(_new))})
    DENSE_RANK_AUDIT = {"n": len(_pairs),
        "exact_order_percent": 100 * sum(x["same_order"] for x in _pairs) / len(_pairs),
        "exact_set_percent": 100 * sum(x["same_set"] for x in _pairs) / len(_pairs),
        "mean_overlap_at_10": sum(x["overlap_at_10"] for x in _pairs) / len(_pairs),
        "rows": _pairs}
    (ARTIFACT_DIR / "dense_ranking_audit.json").write_text(json.dumps(DENSE_RANK_AUDIT, indent=2), encoding="utf-8")
    print("Saved vs regenerated Phase-1 dense retrieval:", {k: v for k, v in DENSE_RANK_AUDIT.items() if k != "rows"})
RANKINGS["hidden"] = build_rankings(questions_hidden, HID_Q_EMB, with_goldmask=False, tag="hidden")

# Both rerankers see exactly the same RRF candidates. Only load one at a time.
if CONFIG["run_rerank"]:
    for _kind in ("bge", "qwen"):
        load_reranker(_kind)
        _system = f"hybrid_{_kind}"
        for _split, _items in (("dev", DEV_ITEMS), ("hidden", questions_hidden)):
            if _split not in RANKINGS or (_split == "hidden" and _system not in CONFIG["hidden_systems"]):
                continue
            _result = {}
            for _i, _item in enumerate(_items):
                _qid = str(_item["qid"])
                _candidates = RERANK_CANDIDATES[_split][_qid]
                _result[_qid] = rerank_search(_item["question"], _candidates, CONFIG["k_output"])
                assert len(_result[_qid]) == CONFIG["k_output"] and len(set(_result[_qid])) == CONFIG["k_output"]
                assert set(_result[_qid]) <= set(_candidates)
                if (_i + 1) % 50 == 0 or _i + 1 == len(_items):
                    print(f"  [{_split}/{_system}] {_i + 1}/{len(_items)}", flush=True)
            RANKINGS[_split][_system] = _result
        unload_reranker()
    (ARTIFACT_DIR / "reranker_timings.json").write_text(json.dumps(RERANK_TIMINGS, indent=2), encoding="utf-8")
    (ARTIFACT_DIR / "reranker_candidates.json").write_text(json.dumps(RERANK_CANDIDATES), encoding="utf-8")
    print("Reranker timings:", RERANK_TIMINGS)

for split, table in RANKINGS.items():
    path = ARTIFACT_DIR / f"rankings_{split}.json"
    path.write_text(json.dumps(table, ensure_ascii=False), encoding="utf-8")
    print(f"saved {path} | systems: {list(table)}")

unload_reranker()

print("\nsample (dev, first question):")
_q0 = str(DEV_ITEMS[0]["qid"])
print(" question:", DEV_ITEMS[0]["question"])
print(" gold    :", gold_ids_of(_q0))
for s in RANKINGS["dev"]:
    print(f" {s:16s}: {RANKINGS['dev'][s][_q0][:6]}")


building rankings: dev=300 questions, hidden=300 questions
  [dev] 50/300 questions  (1s)
  [dev] 100/300 questions  (2s)
  [dev] 150/300 questions  (3s)
  [dev] 200/300 questions  (4s)
  [dev] 250/300 questions  (4s)
  [dev] 300/300 questions  (5s)
Saved vs regenerated Phase-1 dense retrieval: {'n': 300, 'exact_order_percent': 100.0, 'exact_set_percent': 100.0, 'mean_overlap_at_10': 10.0}
  [hidden] 50/300 questions  (1s)
  [hidden] 100/300 questions  (1s)
  [hidden] 150/300 questions  (2s)
  [hidden] 200/300 questions  (3s)
  [hidden] 250/300 questions  (3s)
  [hidden] 300/300 questions  (4s)


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

reranker loaded: qwen Qwen/Qwen3-Reranker-0.6B
  [dev/hybrid_qwen] 50/300
  [dev/hybrid_qwen] 100/300
  [dev/hybrid_qwen] 150/300
  [dev/hybrid_qwen] 200/300
  [dev/hybrid_qwen] 250/300
  [dev/hybrid_qwen] 300/300
Reranker timings: {'bge': {'model': 'BAAI/bge-reranker-v2-m3', 'load_seconds': 19.172957204000113, 'scoring_seconds': 137.3097669840031, 'pairs': 9000}, 'qwen': {'model': 'Qwen/Qwen3-Reranker-0.6B', 'load_seconds': 11.236714048000067, 'scoring_seconds': 328.785071677999, 'pairs': 9000}}
saved /kaggle/working/NLP_phase2/artifacts/rankings_dev.json | systems: ['dense', 'bm25', 'hybrid', 'hybrid_sentence', 'hybrid_goldmask', 'hybrid_bge', 'hybrid_qwen']
saved /kaggle/working/NLP_phase2/artifacts/rankings_hidden.json | systems: ['dense', 'bm25', 'hybrid', 'hybrid_sentence']

sample (dev, first question):
 question: Did the movies Ride A Crooked Trail and The Glassworker, originate from the same country?
 gold    : ['p02917', 'p03365']
 dense           : ['p02917', 'p02916', 'p033

## §8 Context packing

Documents are packed in rank order. The complete candidate context, including IDs, titles and separators, is counted with the LLM tokenizer. Packing stops before the budget is exceeded; full prompt length is checked separately. `retrieved` retains the original ranking.


In [11]:
# =============================================================================
# §8  Context packing under an LLM-token budget
# =============================================================================
llm_tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model"], padding_side="left")
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
print("LLM tokenizer:", CONFIG["llm_model"], "| pad:", llm_tokenizer.pad_token)

DOC_BLOCK = {d: f"[{d}] Title: {DOC_TITLE.get(d, '')}\n{DOC_TEXT.get(d, '')}" for d in DOC_IDS}
BLOCK_SEP_TOKENS = len(llm_tokenizer.encode("\n\n", add_special_tokens=False)) or 1
print("separator token count (LLM tokenizer):", BLOCK_SEP_TOKENS)

# Pre-compute (and cache) the token count of every passage block using the LLM tokenizer.
_tok_cache = ARTIFACT_DIR / "passage_token_counts.json"
if _tok_cache.exists():
    DOC_TOKENS = {k: int(v) for k, v in json.loads(_tok_cache.read_text()).items()}
    print(f"loaded token counts for {len(DOC_TOKENS)} blocks")
else:
    t0 = time.time()
    enc = llm_tokenizer([DOC_BLOCK[d] for d in DOC_IDS], add_special_tokens=False)
    DOC_TOKENS = {d: len(ids) for d, ids in zip(DOC_IDS, enc["input_ids"])}
    _tok_cache.write_text(json.dumps(DOC_TOKENS))
    print(f"tokenized {len(DOC_TOKENS)} blocks in {time.time() - t0:.1f}s")

_tok_vals = np.array(list(DOC_TOKENS.values()))
print(f"passage block tokens: mean={_tok_vals.mean():.0f} max={_tok_vals.max()} "
      f"p95={np.percentile(_tok_vals, 95):.0f}")


def pack_context(doc_ids, budget=None, k_context=None, policy="stop"):
    """Greedy packing of passages in rank order up to `budget` LLM tokens.

    Parameters
    ----------
    doc_ids : iterable of doc ids (in rank order, best first).
    budget  : max tokens for the concatenated document blocks (default from CONFIG).
    k_context : max number of passages considered.
    policy  : "stop" -> break at the first overflow; "skip" -> drop the overflowing
              passage and keep going.

    Returns a dict with: ids, text, n_tokens, dropped, skipped.
    """
    budget = int(budget if budget is not None else CONFIG["max_context_tokens"])
    k_context = int(k_context if k_context is not None else CONFIG["k_context"])
    assert policy in ("stop", "skip"), "policy must be 'stop' or 'skip'"

    chosen, used, blocks, skipped = [], 0, [], []
    for d in list(doc_ids)[:k_context]:
        candidate_text = "\n\n".join(blocks + [DOC_BLOCK[d]])
        candidate_tokens = len(llm_tokenizer.encode(candidate_text, add_special_tokens=False))
        if candidate_tokens > budget:
            if policy == "stop":
                break
            skipped.append(d)
            continue
        chosen.append(d)
        used = candidate_tokens
        blocks.append(DOC_BLOCK[d])

    dropped = [d for d in list(doc_ids)[:k_context] if d not in chosen]
    return {
        "ids": chosen,
        "text": "\n\n".join(blocks),
        "n_tokens": used,
        "dropped": dropped,
        "skipped": skipped,
    }


# -------------- self-tests
_pick = RANKINGS["dev"]["hybrid"][str(DEV_ITEMS[0]["qid"])]
_pk = pack_context(_pick, budget=10**6)
assert _pk["ids"] == _pick, "rank order must be preserved"
assert _pk["n_tokens"] == len(llm_tokenizer.encode(_pk["text"], add_special_tokens=False))

_pk_def = pack_context(_pick)
assert _pk_def["n_tokens"] <= CONFIG["max_context_tokens"]

_tight = pack_context(_pick, budget=60)
assert _tight["n_tokens"] <= 60, f"budget violated: {_tight['n_tokens']}"
assert len(_tight["ids"]) < len(_pick), "tight budget must drop passages"

_skip = pack_context(_pick, budget=60, policy="skip")
assert _skip["n_tokens"] <= 60
assert len(_skip["skipped"]) > 0 or len(_skip["ids"]) < 2

_empty = pack_context([], budget=100)
assert _empty["ids"] == [] and _empty["text"] == "" and _empty["n_tokens"] == 0

assert pack_context(_pick, budget=0)["ids"] == []

print("\npacking self-checks passed.")
print(f"sample: {len(_pk_def['ids'])} docs, {_pk_def['n_tokens']} tokens "
      f"(budget {CONFIG['max_context_tokens']})")
print(f"tight : {len(_tight['ids'])} docs, {_tight['n_tokens']} tokens (budget 60)")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

LLM tokenizer: Qwen/Qwen3-8B | pad: <|endoftext|>
separator token count (LLM tokenizer): 1
tokenized 3882 blocks in 0.5s
passage block tokens: mean=127 max=1255 p95=371

packing self-checks passed.
sample: 10 docs, 757 tokens (budget 1000)
tight : 0 docs, 0 tokens (budget 60)


## §9 Language model and answer parsing

The generator (`Qwen/Qwen3-8B`) is prompted with a strict template that enforces:

- A short (1–6 word) final answer on the very last line: `Answer: <short answer>`.
- Exact `Answer: Yes` / `Answer: No` for boolean questions.
- Inline citations in square brackets using **only** the document IDs that appear in the `Documents:` block.
- `INSUFFICIENT_INFORMATION` (and no `Answer:` line) when evidence is missing.

A dedicated parser strips any `<think>…</think>` blocks, extracts citations, finds the last `Answer:` line, normalizes the answer text, maps pure-ID answers to document titles, and treats empty / unknown answers as abstentions.


In [12]:
# =============================================================================
# §9  Language model, prompt, batched generation, output parser
# =============================================================================
SYSTEM_PROMPT = ("You are a precise question-answering system. You answer only from the "
                 "provided documents, cite them, and admit when the evidence is missing.")

USER_TEMPLATE = """Answer the question using ONLY the documents below. Do not use outside knowledge.

Rules:
1. You may reason briefly, but the VERY LAST line of your reply must be exactly:
Answer: <short answer>
2. <short answer> is 1-6 words: a title, a name, a place, a date, a number, or Yes/No. Never a full sentence.
3. For yes/no questions write exactly "Answer: Yes" or "Answer: No".
4. Cite every supporting document inline with its id in square brackets, e.g. [p00012].
   Use only ids that appear in the Documents block. Never invent ids.
5. If the documents do not contain enough information to answer, reply with exactly:
INSUFFICIENT_INFORMATION
and do not write an Answer line.
6. If several candidates seem plausible, commit to exactly one. Never write "X or Y".

Documents:
{context}

Question: {question}

Answer:
"""

PROMPT_FILE = PROMPT_DIR / "phase2_strict.txt"
PROMPT_FILE.write_text(SYSTEM_PROMPT + "\n\n" + USER_TEMPLATE, encoding="utf-8")
print("prompt saved to:", PROMPT_FILE)

# ---------------------------------------------------------------- load LLM
torch.cuda.empty_cache()
gc.collect()
print("loading LLM:", CONFIG["llm_model"])
for i in range(torch.cuda.device_count()):
    free_m, total_m = torch.cuda.mem_get_info(i)
    print(f"  GPU {i} free before LLM load: {free_m / 1024**3:.2f} GiB / {total_m / 1024**3:.2f} GiB")

llm_tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model"], padding_side="left")
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_model = load_causal(
    AutoModelForCausalLM, CONFIG["llm_model"],
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=resolve_max_memory(),
)
llm_model.eval()
print("device map:", getattr(llm_model, "hf_device_map", "single device"))
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1024**3
    free_m, total_m = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: allocated {alloc:.2f} GiB | free {free_m / 1024**3:.2f} GiB")


def apply_chat(system, user):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    try:
        return llm_tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return llm_tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)


def _generate_once(prompts, max_new_tokens):
    """Single greedy generation pass (no retry). Caller handles OOM.

    Uses the Phase-2 batched greedy generation adapter. The original
    Phase-1 generator is single-sample and has a different prompt.
    """
    return generate_from_prompts(
        prompts, llm_tokenizer, llm_model,
        max_new_tokens=max_new_tokens,
        max_length=CONFIG["max_prompt_tokens"],
        pad_token_id=llm_tokenizer.pad_token_id)


@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=None):
    """Greedy generation with automatic OOM fallback.

    Tries the full batch first; on OutOfMemoryError it clears the cache and
    retries by splitting the batch in half (recursively down to batch=1).
    Minor floating-point differences across batch sizes are possible.
    """
    max_new_tokens = int(max_new_tokens or CONFIG["max_new_tokens"])
    try:
        return _generate_once(prompts, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        if len(prompts) <= 1:
            raise
        mid = len(prompts) // 2
        print(f"  [OOM] batch {len(prompts)} -> retrying as {mid}+{len(prompts)-mid}",
              flush=True)
        return (generate_batch(prompts[:mid], max_new_tokens)
                + generate_batch(prompts[mid:], max_new_tokens))


# ---------------------------------------------------------------- output parser
ABSTAIN_TOKEN = "INSUFFICIENT_INFORMATION"
_ID_RE = re.compile(r"\bp\d{4,6}\b", flags=re.IGNORECASE)
_CITE_RE = re.compile(r"\[\s*(p\d{4,6})\s*\]", flags=re.IGNORECASE)


def strip_thinking(text):
    text = re.sub(r"<think(?:ing)?>.*?</think(?:ing)?>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    # An unclosed thinking block is discarded rather than treated as an answer.
    text = re.sub(r"<think(?:ing)?>.*$", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"</?think(?:ing)?>", " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


def parse_llm_output(raw, context_ids):
    """Convert raw model output into a structured dict.

    Returns dict(answer, cited, abstained) where:
      - cited contains only ids that actually appear in the context.
      - abstained=True means the model refused / did not produce an answer.
      - A bare document id is not a valid answer.

    The answer text is extracted with the Phase-2 extract_short_answer adapter
    (cell 15) — no internal copy.
    """
    context_ids = [str(c) for c in context_ids]
    ctx_set = set(context_ids)
    text = strip_thinking(raw)

    cited = list(dict.fromkeys(m.lower() for m in _CITE_RE.findall(text) if m.lower() in ctx_set))
    # Accept JSON output as a fallback, without treating arbitrary IDs as citations.
    _json_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE)
    try:
        obj = json.loads(_json_text)
    except (ValueError, TypeError):
        obj = None
    if isinstance(obj, dict):
        if obj.get("abstained") is True:
            return {"answer": "", "cited": [], "abstained": True}
        value = obj.get("answer", "")
        answer = value.strip() if isinstance(value, str) else ""
        refs = obj.get("cited", obj.get("citations", []))
        if not isinstance(refs, list):
            refs = []
        cited = list(dict.fromkeys(str(c).lower().strip(" []") for c in refs
                                   if str(c).lower().strip(" []") in ctx_set))
    else:
        if ABSTAIN_TOKEN.lower() in text.lower():
            return {"answer": "", "cited": [], "abstained": True}
        answer = extract_short_answer(text).strip("\"'“”«»")
    if not answer:
        return {"answer": "", "cited": [], "abstained": True}

    if _ID_RE.fullmatch(answer.strip()) or answer.lower() in {"insufficient information", "unknown", "n a"}:
        return {"answer": "", "cited": [], "abstained": True}

    assert set(cited) <= ctx_set, "cited ids must come from the packed context"
    return {"answer": answer, "cited": cited, "abstained": False}


# -------------- parser self-tests
_ctx = ["p00001", "p00002"]
assert parse_llm_output("Answer: Yes", _ctx)["answer"] == "Yes"
assert parse_llm_output("blah blah\nAnswer: Paris [p00001]", _ctx) == \
    {"answer": "Paris", "cited": ["p00001"], "abstained": False}
assert parse_llm_output("Evidence [p00009] says X. Answer: X", _ctx)["cited"] == [], \
    "citations outside the context must be dropped"
assert parse_llm_output("INSUFFICIENT_INFORMATION", _ctx) == \
    {"answer": "", "cited": [], "abstained": True}
assert parse_llm_output("<think>long reasoning</think>Answer: No", _ctx)["answer"] == "No"
assert parse_llm_output("<think>unclosed reasoning", _ctx)["abstained"] is True
assert parse_llm_output("Answer: Paris [p00001] [p00001] [p00009]", _ctx)["cited"] == ["p00001"]
assert parse_llm_output("Answer: p00002", _ctx)["cited"] == []
assert parse_llm_output("Answer: p00002", _ctx)["abstained"] is True
assert parse_llm_output('{"answer":"Paris","cited":["p00001","p00009"],"abstained":false}', _ctx) == {"answer":"Paris", "cited":["p00001"], "abstained":False}
assert parse_llm_output('{"answer":"Paris","abstained":true}', _ctx)["abstained"] is True
assert parse_llm_output("Answer: ", _ctx)["abstained"] is True
assert parse_llm_output("Answer: The answer is \"Paris\".", _ctx)["answer"] == "Paris"
print("parser self-checks passed")

# Quick smoke generation on the first dev question.
_smoke_q = questions_dev[0]["question"]
_smoke_ctx = pack_context(RANKINGS["dev"]["hybrid"][str(questions_dev[0]["qid"])])["ids"]
_smoke_prompt = apply_chat(SYSTEM_PROMPT, USER_TEMPLATE.format(
    context=pack_context(_smoke_ctx)["text"], question=_smoke_q))
_smoke_out = generate_batch([_smoke_prompt])[0]
print("\nsmoke generation:")
print("--- raw ---")
print(_smoke_out[:600])
print("--- parsed ---")
print(parse_llm_output(_smoke_out, _smoke_ctx))


prompt saved to: /kaggle/working/NLP_phase2/prompts/phase2_strict.txt
loading LLM: Qwen/Qwen3-8B
  GPU 0 free before LLM load: 14.41 GiB / 14.56 GiB
  GPU 1 free before LLM load: 14.41 GiB / 14.56 GiB
max_memory (int keys): {0: '12GiB', 1: '12GiB', 'cpu': '40GiB'}


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}
  GPU 0: allocated 7.64 GiB | free 6.77 GiB
  GPU 1: allocated 7.64 GiB | free 6.77 GiB
parser self-checks passed

smoke gener

## §10 Dev runs

Generate and save hybrid, sentence-level hybrid, optional reranker and gold-mask runs. Each has predictions and a configuration. The Phase-1 dense predictions are reused without regeneration; its original prompt and budget may differ from Phase 2.


In [13]:
# =============================================================================
# §10  Run all dev systems, save predictions and run_config snapshots
# =============================================================================
import yaml

# Phase-1 helpers are reused directly — there are no internal copies.
REQUIRED_FIELDS = PHASE1_UTILS.REQUIRED_FIELDS
_save_predictions_p1 = PHASE1_UTILS.save_predictions


def save_predictions(records, path):
    _save_predictions_p1(records, path)
    print(f"  saved {len(records)} predictions -> {path}")


CONTEXT_USED = {}      # system -> {qid: [context ids actually packed]}
RAW_OUTPUTS = {}       # system -> {qid: raw LLM text}


def write_run_config(system, split, path, extra=None):
    cfg = {
        "system_name": system,
        "phase": 2,
        "passage_embeddings_source": PASSAGE_EMB_ORIGIN,
        "split": split,
        "description": {
            "dense": "Phase-1 dense retrieval recipe regenerated with Phase-2 prompt, packing and generation settings (controlled baseline)",
            "hybrid": "Dense + hand-written BM25 fused with RRF, context packing, citations, abstention",
            "hybrid_sentence": "Sentence-level dense retrieval (max-pool) fused with passage BM25 via RRF",
            "hybrid_bge": "Hybrid RRF candidates reranked by BGE classifier",
            "hybrid_qwen": "Same hybrid RRF candidates reranked by Qwen causal yes/no logits",
            "hybrid_goldmask": "Hybrid retrieval with gold documents masked (abstention experiment)",
        }.get(system, system),
        "models": {
            "llm": {"name": CONFIG["llm_model"], "where": "Kaggle, 2x Tesla T4, transformers fp16",
                    "temperature": CONFIG["temperature"]},
            "embedder": {"name": CONFIG["embedding_model"],
                         "max_sequence_length": CONFIG["embed_max_len"]},
            "reranker": {"hybrid_bge": CONFIG["bge_reranker_model"], "hybrid_qwen": CONFIG["qwen_reranker_model"]}.get(system),
        },
        "settings": {
            "retrieval": {
                "unit": "sentence" if system == "hybrid_sentence" else "passage",
                "dense_depth": CONFIG["fusion_depth"],
                "bm25_depth": CONFIG["fusion_depth"],
                "fusion": "rrf" if system.startswith("hybrid") else None,
                "rrf_k": CONFIG["rrf_k"],
                "bm25": {"k1": CONFIG["bm25_k1"], "b": CONFIG["bm25_b"],
                         "stopwords_removed": CONFIG["bm25_remove_stopwords"]},
                "output_k": CONFIG["k_output"],
                "context_k": CONFIG["k_context"],
                "rerank_candidates": CONFIG["rerank_candidates"] if system in ("hybrid_bge", "hybrid_qwen") else None,
                "rerank_max_length": CONFIG["rerank_max_length"] if system in ("hybrid_bge", "hybrid_qwen") else None,
                "rerank_batch": CONFIG["bge_rerank_batch"] if system == "hybrid_bge" else CONFIG["qwen_rerank_batch"] if system == "hybrid_qwen" else None,
                "rerank_instruction": QWEN_RERANK_INSTRUCTION if system == "hybrid_qwen" else None,
                "document_mask": system == "hybrid_goldmask",
            },
            "context_packing": {"max_context_tokens": CONFIG["max_context_tokens"],
                                "policy": "stop", "token_counter": "llm_tokenizer"},
            "max_prompt_tokens": CONFIG["max_prompt_tokens"],
            "max_new_tokens": CONFIG["max_new_tokens"],
            "llm_batch": CONFIG["llm_batch"],
            "do_sample": False,
            "no_repeat_ngram_size": 8,
            "repetition_penalty": 1.05,
            "prompt_file": "prompts/phase2_strict.txt",
        },
        "command": "jupyter nbconvert --execute notebook.ipynb (sections §1-§15)",
        "date": str(date.today()),
    }
    if extra:
        cfg["settings"].update(extra)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")


def run_system(system, split, items, ranking_table, save=True, desc=""):
    """Generate answers for every question using the given ranking table."""
    records, ctx_map, raw_map = [], {}, {}
    t0 = time.time()
    n = len(items)
    batch = max(1, CONFIG["llm_batch"])

    for start in range(0, n, batch):
        chunk = items[start:start + batch]
        prompts, ctxs = [], []
        for it in chunk:
            qid = str(it["qid"])
            packed = pack_context(ranking_table[qid])
            ctxs.append(packed["ids"])
            prompt = apply_chat(SYSTEM_PROMPT, USER_TEMPLATE.format(
                context=packed["text"], question=it["question"]))
            if len(llm_tokenizer(prompt, add_special_tokens=True)["input_ids"]) > CONFIG["max_prompt_tokens"]:
                raise ValueError(f"Prompt for {qid} exceeds max_prompt_tokens; reduce the context budget.")
            prompts.append(prompt)
        raws = generate_batch(prompts)

        for it, raw, ctx in zip(chunk, raws, ctxs):
            qid = str(it["qid"])
            res = parse_llm_output(raw, ctx)
            records.append({
                "qid": qid,
                "answer": res["answer"],
                "retrieved": list(ranking_table[qid]),
                "cited": res["cited"],
                "abstained": res["abstained"],
            })
            ctx_map[qid] = ctx
            raw_map[qid] = raw

        done = start + len(chunk)
        if done % 50 < batch or done == n:
            rate = done / max(time.time() - t0, 1e-9)
            eta = (n - done) / max(rate, 1e-9)
            print(f"  [{system}] {done}/{n}  ({time.time() - t0:.0f}s, ETA {eta:.0f}s)",
                  flush=True)

    CONTEXT_USED[system] = ctx_map
    RAW_OUTPUTS[system] = raw_map

    if save:
        out_dir = RUNS_DIR / split / system
        save_predictions(records, out_dir / "predictions.jsonl")
        write_run_config(system, split, out_dir / "run_config.yaml")
        (ARTIFACT_DIR / f"context_used_{split}_{system}.json").write_text(
            json.dumps(ctx_map), encoding="utf-8")
        (ARTIFACT_DIR / f"raw_outputs_{split}_{system}.json").write_text(
            json.dumps(raw_map, ensure_ascii=False), encoding="utf-8")
    return records


# ---------------------------------------------------------------- dev runs
DEV_SYSTEMS = []
if CONFIG["run_dev"]:
    if CONFIG["run_dense_baseline"]:
        DEV_SYSTEMS.append("dense")
    DEV_SYSTEMS.append("hybrid")
    if CONFIG["run_sentence"]:
        DEV_SYSTEMS.append("hybrid_sentence")
    if CONFIG["run_rerank"]:
        DEV_SYSTEMS.extend(["hybrid_bge", "hybrid_qwen"])
    if CONFIG["run_goldmask"]:
        DEV_SYSTEMS.append("hybrid_goldmask")

DEV_PREDS = {}
for system in DEV_SYSTEMS:
    print(f"\n=== dev / {system} ===")
    DEV_PREDS[system] = run_system(system, "dev", DEV_ITEMS, RANKINGS["dev"][system])

print("\ncompleted dev systems:", list(DEV_PREDS))



=== dev / dense ===
  [dense] 50/300  (191s, ETA 954s)
  [dense] 100/300  (404s, ETA 809s)
  [dense] 150/300  (613s, ETA 613s)
  [dense] 200/300  (817s, ETA 409s)
  [dense] 250/300  (1023s, ETA 205s)
  [dense] 300/300  (1234s, ETA 0s)
Saved 300 predictions -> /kaggle/working/NLP_phase2/runs/dev/dense/predictions.jsonl
  saved 300 predictions -> /kaggle/working/NLP_phase2/runs/dev/dense/predictions.jsonl

=== dev / hybrid ===
  [hybrid] 50/300  (191s, ETA 957s)
  [hybrid] 100/300  (401s, ETA 802s)
  [hybrid] 150/300  (599s, ETA 599s)
  [hybrid] 200/300  (799s, ETA 399s)
  [hybrid] 250/300  (1004s, ETA 201s)
  [hybrid] 300/300  (1204s, ETA 0s)
Saved 300 predictions -> /kaggle/working/NLP_phase2/runs/dev/hybrid/predictions.jsonl
  saved 300 predictions -> /kaggle/working/NLP_phase2/runs/dev/hybrid/predictions.jsonl

=== dev / hybrid_sentence ===
  [hybrid_sentence] 50/300  (195s, ETA 973s)
  [hybrid_sentence] 100/300  (401s, ETA 803s)
  [hybrid_sentence] 150/300  (605s, ETA 605s)
  [hybr

## §11 Scoring

Use the course `score.py` for dev metrics and official paired differences against Phase-1 dense. Within-type paired differences use the same `per_question` and `bootstrap` functions from that script.


In [14]:
# =============================================================================
# §11  Scoring with score.py and paired comparisons
# =============================================================================
# Scoring helpers adapt score.py output for Phase 2.
SCORES = {}          # system -> parsed score.py output
PAIRED = {}          # "A__vs__B" -> paired metrics
# SCORE_PATH was resolved in §1 (either in PACK_DIR or the dataset root).
assert SCORE_PATH.is_file(), f"score.py not found: {SCORE_PATH}"
print("using score.py:", SCORE_PATH)


def run_score(pred_path, split="dev", compare=None, partial=False, validate_only=False):
    return PHASE1_UTILS.run_score(pred_path, SCORE_PATH, PACK_DIR, split=split,
                                  compare=compare, partial=partial,
                                  validate_only=validate_only)


parse_score_output = PHASE1_UTILS.parse_score_output


def pred_path(system, split="dev"):
    return RUNS_DIR / split / system / "predictions.jsonl"


# When dev_limit is set, score.py must be invoked with --partial.
PARTIAL_DEV = CONFIG["dev_limit"] is not None

if CONFIG["run_dev"]:
    for system in DEV_SYSTEMS:
        print(f"\n{'=' * 20} {system} {'=' * 20}")
        text_out = run_score(pred_path(system), "dev", partial=PARTIAL_DEV)
        print(text_out)
        SCORES[system] = parse_score_output(text_out)
        (ARTIFACT_DIR / f"score_{system}.txt").write_text(text_out, encoding="utf-8")

    # ---- Official Phase-1 dense baseline (if attached) ----
    # The project asks for a comparison against the Phase-1 dense system. When a
    # Phase-1 predictions.jsonl is discovered, it is copied verbatim and scored
    # with the same score.py so the paired comparison is exact.
    _src = PHASE1_ARTIFACTS["NLP_phase1_output/dev/dense/predictions.jsonl"]
    if _src:
        _dst_dir = RUNS_DIR / "dev" / "phase1_dense"
        _dst_dir.mkdir(parents=True, exist_ok=True)
        _rows = [r for r in read_jsonl(_src) if str(r["qid"]) in QDEV_BY_QID]
        if CONFIG["dev_limit"]:
            _allowed = {str(q["qid"]) for q in DEV_ITEMS}
            _rows = [r for r in _rows if str(r["qid"]) in _allowed]
        save_predictions(_rows, _dst_dir / "predictions.jsonl")
        _original_config = PHASE1_ARTIFACTS["NLP_phase1_output/dev/dense/run_config.yaml"]
        if _original_config is None:
            raise FileNotFoundError("Phase-1 dense run_config.yaml is required to preserve baseline provenance.")
        shutil.copy2(_original_config, _dst_dir / "run_config.yaml")
        print("\nPhase-1 dense baseline found:", _src)
        _txt = run_score(pred_path("phase1_dense"), "dev", partial=PARTIAL_DEV)
        print(_txt)
        SCORES["phase1_dense"] = parse_score_output(_txt)
        (ARTIFACT_DIR / "score_phase1_dense.txt").write_text(_txt, encoding="utf-8")
    else:
        raise FileNotFoundError("Phase-1 NLP_phase1_output/dev/dense/predictions.jsonl is required for the paired baseline comparison.")

    # ---- Paired comparisons ----
    comparisons = []
    baseline = "dense" if "dense" in DEV_SYSTEMS else None
    if baseline:
        for system in DEV_SYSTEMS:
            if system != baseline:
                comparisons.append((system, baseline))
    if "hybrid" in DEV_SYSTEMS:
        for system in ("hybrid_sentence", "hybrid_bge", "hybrid_qwen"):
            if system in DEV_SYSTEMS and (system, "hybrid") not in comparisons:
                comparisons.append((system, "hybrid"))
    if {"hybrid_bge", "hybrid_qwen"} <= set(DEV_SYSTEMS):
        comparisons.append(("hybrid_qwen", "hybrid_bge"))
    if "phase1_dense" in SCORES and "hybrid" in DEV_SYSTEMS:
        comparisons.append(("hybrid", "phase1_dense"))

    for system, other in comparisons:
        print(f"\n{'=' * 12} paired: {system} minus {other} {'=' * 12}")
        text_out = run_score(pred_path(system), "dev",
                             compare=pred_path(other), partial=PARTIAL_DEV)
        print(text_out)
        PAIRED[f"{system}__vs__{other}"] = parse_score_output(text_out)["paired"]
        (ARTIFACT_DIR / f"paired_{system}_vs_{other}.txt").write_text(text_out, encoding="utf-8")

    # Paired within-type differences against controlled dense use score.py functions.
    import importlib.util as _score_import
    _score_spec = _score_import.spec_from_file_location("course_score", SCORE_PATH)
    _course_score = _score_import.module_from_spec(_score_spec)
    _score_spec.loader.exec_module(_course_score)
    _a = {str(x["qid"]): x for x in read_jsonl(pred_path("hybrid"))}
    if "dense" not in DEV_SYSTEMS:
        raise RuntimeError("Controlled dense baseline is required for fair paired per-type comparisons.")
    _b = {str(x["qid"]): x for x in read_jsonl(pred_path("dense"))}
    if set(_a) != set(_b):
        raise ValueError("Paired comparison requires identical hybrid and controlled dense qids.")
    PAIRED_BY_TYPE = {}
    for _qid in sorted(_a):
        _lab = LABELS_BY_QID[_qid]
        _type = _lab["type"]
        _ma, _mb = _course_score.per_question(_a[_qid], _lab), _course_score.per_question(_b[_qid], _lab)
        _bucket = PAIRED_BY_TYPE.setdefault(_type, {})
        for _metric in _ma.keys() & _mb.keys():
            _bucket.setdefault(_metric, []).append(_ma[_metric] - _mb[_metric])
    PAIRED_BY_TYPE = {
        t: {m: {"value": 100 * sum(v) / len(v),
                 "ci": tuple(100 * x for x in _course_score.bootstrap(v)), "n": len(v)}
            for m, v in metrics.items()}
        for t, metrics in PAIRED_BY_TYPE.items()
    }
    (ARTIFACT_DIR / "paired_by_type.json").write_text(
        json.dumps(PAIRED_BY_TYPE, ensure_ascii=False, indent=2), encoding="utf-8")

    # ---- Summary table ----
    import pandas as pd

    rows = []
    for system in SCORES:
        ov = SCORES[system]["overall"]
        rows.append({
            "system": system,
            "n": SCORES[system]["overall_n"],
            **{k: round(ov[k]["value"], 1) for k in
               ("recall@10", "hitall@10", "mrr", "ndcg@10", "em", "f1",
                "cite_precision", "cite_recall", "abstained") if k in ov},
        })
    SUMMARY_DEV = pd.DataFrame(rows).set_index("system")
    print("\n=== DEV SUMMARY (percent, from score.py) ===")
    display(SUMMARY_DEV)

    summary_export = {
        "scores": {
            system: {
                "overall": {m: v["value"] for m, v in s["overall"].items()},
                "by_type": {t: {m: v["value"] for m, v in s["by_type"][t].items()}
                            for t in s["by_type"]},
                "type_n": s["type_n"],
                "n": s["overall_n"],
            }
            for system, s in SCORES.items()
        },
        "paired": {k: {m: v["value"] for m, v in p.items()} for k, p in PAIRED.items()},
    }
    (ARTIFACT_DIR / "summary_dev.json").write_text(
        json.dumps(summary_export, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", ARTIFACT_DIR / "summary_dev.json")


using score.py: /kaggle/input/datasets/mohammadhossein20/nlp-phase1-dataset/course_pack_v1/course_pack_v1/score.py

==================== dense ====================
valid: /kaggle/working/NLP_phase2/runs/dev/dense/predictions.jsonl (300 predictions)

overall (n=300)
  recall@2          65.5   [ 62.6,  68.0]
  hitall@2          32.0   [ 26.3,  37.0]
  recall@5          75.4   [ 72.7,  78.0]
  hitall@5          48.3   [ 42.7,  54.0]
  recall@10         79.2   [ 76.5,  81.7]
  hitall@10         55.3   [ 49.3,  60.7]
  mrr               98.8   [ 98.0,  99.7]
  ndcg@10           80.6   [ 78.7,  82.4]
  em                30.7   [ 25.3,  36.3]
  f1                39.8   [ 34.7,  44.9]
  cite_precision    49.1   [ 43.4,  54.9]
  cite_recall       33.1   [ 28.7,  37.2]
  abstained         38.3   [ 33.0,  44.0]

type: bridge_comparison (n=66)
  recall@2          47.7   [ 45.8,  49.2]
  hitall@2           0.0   [  0.0,   0.0]
  recall@5          55.7   [ 53.4,  58.0]
  hitall@5           0.0   [  

,n,recall@10,hitall@10,mrr,ndcg@10,em,f1,cite_precision,cite_recall,abstained
system,,,,,,,,,,
dense,300,79.2,55.3,98.8,80.6,30.7,39.8,49.1,33.1,38.3
hybrid,300,73.2,44.3,95.2,74.7,29.3,37.0,47.0,33.2,41.0
hybrid_sentence,300,73.3,44.0,95.6,74.9,28.3,35.8,45.5,31.8,41.3
hybrid_bge,300,78.1,53.7,99.2,80.3,27.7,37.6,50.6,34.7,35.3
hybrid_qwen,300,77.3,51.3,98.3,79.4,29.3,38.0,46.6,33.0,39.7
hybrid_goldmask,300,0.0,0.0,0.0,0.0,0.7,2.3,0.0,0.0,87.0
phase1_dense,300,79.2,55.3,98.8,80.6,37.0,46.5,52.7,36.0,35.0


saved: /kaggle/working/NLP_phase2/artifacts/summary_dev.json


## §12 Gold-mask abstention

Measure masked abstention, correct and wrong answers separately, and normal-run over-abstention when all gold IDs are in the packed context. Removing gold IDs does not prove that alternative evidence is absent.


In [15]:
# =============================================================================
# §12  Gold-mask abstention analysis: grounded vs parametric vs hallucination vs over-abstention
# =============================================================================

def normalize_answer(s):
    s = "".join(ch for ch in str(s).lower() if ch not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())


def is_correct(pred, label):
    if pred.get("abstained"):
        return False
    golds = [label["answer"]] + list(label.get("aliases", []))
    return any(normalize_answer(pred["answer"]) == normalize_answer(g) for g in golds)


ABSTENTION_ROWS = []
if CONFIG["run_dev"] and "hybrid" in DEV_PREDS and "hybrid_goldmask" in DEV_PREDS:
    normal = {r["qid"]: r for r in DEV_PREDS["hybrid"]}
    masked = {r["qid"]: r for r in DEV_PREDS["hybrid_goldmask"]}

    for item in DEV_ITEMS:
        qid = str(item["qid"])
        label = LABELS_BY_QID[qid]
        gold = set(gold_ids_of(qid))
        ctx_normal = set(CONTEXT_USED["hybrid"].get(qid, []))

        p_n, p_m = normal[qid], masked[qid]
        n_ok, m_ok = is_correct(p_n, label), is_correct(p_m, label)

        # Mutually exclusive observed outcomes, not proof of evidence absence.
        if p_m["abstained"]:
            cat = "masked_abstained"
        elif m_ok:
            cat = "masked_correct"
        else:
            cat = "masked_wrong"

        ABSTENTION_ROWS.append({
            "qid": qid,
            "type": item.get("type", "unknown"),
            "gold_in_context": float(gold <= ctx_normal),
            "normal_abstained": float(p_n["abstained"]),
            "normal_correct": float(n_ok),
            "normal_over_abstained": float(bool(p_n["abstained"]) and gold <= ctx_normal),
            "masked_abstained": float(p_m["abstained"]),
            "masked_correct": float(m_ok),
            "category": cat,
        })

    import pandas as pd
    ABST_DF = pd.DataFrame(ABSTENTION_ROWS)
    ABST_DF.to_csv(ARTIFACT_DIR / "abstention_analysis.csv", index=False)

    dist = ABST_DF["category"].value_counts()
    total = len(ABST_DF)
    print("=== gold-mask abstention experiment (dev) ===")
    for cat, n in dist.items():
        print(f"  {cat:22s} {n:4d}  ({n / total:5.1%})")
    print(f"\n  normal abstention rate : {ABST_DF['normal_abstained'].mean():.1%}")
    print(f"  masked abstention rate : {ABST_DF['masked_abstained'].mean():.1%}")
    print(f"  masked wrong answers   : {(ABST_DF.category == 'masked_wrong').sum()} "
          f"({(ABST_DF.category == 'masked_wrong').mean():.1%})")
    print(f"  masked correct (evidence may exist): {(ABST_DF.category == 'masked_correct').sum()} "
          f"({(ABST_DF.category == 'masked_correct').mean():.1%})")

    ABST_SUMMARY = {
        "category_counts": dist.to_dict(),
        "normal_abstention_rate": float(ABST_DF["normal_abstained"].mean()),
        "masked_abstention_rate": float(ABST_DF["masked_abstained"].mean()),
        "masked_wrong_rate": float((ABST_DF.category == "masked_wrong").mean()),
        "masked_correct_rate": float((ABST_DF.category == "masked_correct").mean()),
        "over_abstention_rate": float(ABST_DF["normal_over_abstained"].mean()),
        "n": total,
    }
    (ARTIFACT_DIR / "abstention_summary.json").write_text(
        json.dumps(ABST_SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\nPer-question-type abstention rates (gold-masked run):")
    print(ABST_DF.groupby("type")[["masked_abstained", "masked_correct"]].mean().round(3))
else:
    ABST_DF, ABST_SUMMARY = None, {}
    print("gold-mask experiment skipped (run_goldmask=False or no dev run)")


=== gold-mask abstention experiment (dev) ===
  masked_abstained        261  (87.0%)
  masked_wrong             37  (12.3%)
  masked_correct            2  ( 0.7%)

  normal abstention rate : 41.0%
  masked abstention rate : 87.0%
  masked wrong answers   : 37 (12.3%)
  masked correct (evidence may exist): 2 (0.7%)

Per-question-type abstention rates (gold-masked run):
                   masked_abstained  masked_correct
type                                               
bridge_comparison             0.955           0.000
comparison                    0.877           0.000
compositional                 0.872           0.016
inference                     0.694           0.000


## §13 Additional analyses

Results by type, bridge-comparison metrics and a descriptive error taxonomy are calculated from this run. Examples and supporting IDs are saved to `error_analysis.txt`.


In [16]:
# =============================================================================
# §13  Per-type analysis, bridge bottleneck, and error taxonomy
# =============================================================================
import pandas as pd

TYPE_ORDER = ["comparison", "bridge_comparison", "compositional", "inference"]
TYPE_EN = {"comparison": "comparison", "bridge_comparison": "bridge",
           "compositional": "compositional", "inference": "inference"}

if CONFIG["run_dev"]:
    # ---- per-type metrics table ----
    rows = []
    for system in DEV_SYSTEMS:
        for qtype in TYPE_ORDER:
            if qtype not in SCORES[system]["by_type"]:
                continue
            m = SCORES[system]["by_type"][qtype]
            rows.append({
                "system": system, "type": qtype, "n": SCORES[system]["type_n"].get(qtype, 0),
                **{k.replace("@", "_"): round(m[k]["value"], 1) for k in
                   ("recall@10", "hitall@10", "mrr", "ndcg@10", "em", "f1",
                    "cite_precision", "cite_recall", "abstained") if k in m},
            })
    BY_TYPE_DF = pd.DataFrame(rows)
    print("=== metrics by question type ===")
    display(BY_TYPE_DF.round(1))

    # ---- bridge bottleneck ----
    print("\n=== bridge_comparison bottleneck ===")
    bridge_rows = []
    for system in DEV_SYSTEMS:
        bt = SCORES[system]["by_type"].get("bridge_comparison")
        if not bt:
            continue
        bridge_rows.append({
            "system": system,
            "n": SCORES[system]["type_n"]["bridge_comparison"],
            "recall@2": bt["recall@2"]["value"], "recall@5": bt["recall@5"]["value"],
            "recall@10": bt["recall@10"]["value"],
            "hitall@2": bt["hitall@2"]["value"], "hitall@5": bt["hitall@5"]["value"],
            "hitall@10": bt["hitall@10"]["value"],
            "mrr": bt["mrr"]["value"], "em": bt["em"]["value"],
        })
    BRIDGE_DF = pd.DataFrame(bridge_rows).set_index("system").round(1)
    display(BRIDGE_DF)
    avg_gold_bridge = np.mean([len(gold_ids_of(str(q["qid"]))) for q in DEV_ITEMS
                               if q.get("type") == "bridge_comparison"])
    print(f"average gold documents per bridge question: {avg_gold_bridge:.2f}")

    # ---- error taxonomy ----
    ERR = []
    hybrid = {r["qid"]: r for r in DEV_PREDS["hybrid"]}
    for item in DEV_ITEMS:
        qid = str(item["qid"])
        label = LABELS_BY_QID[qid]
        gold = set(gold_ids_of(qid))
        ctx = set(CONTEXT_USED["hybrid"].get(qid, []))
        rec = RANKINGS["dev"]["hybrid"][qid]
        pred = hybrid[qid]
        ok = is_correct(pred, label)

        if not (gold <= set(rec[:10])):
            err = "retrieval_failure"
        elif not (gold <= ctx):
            err = "packing_evidence_gap"
        elif not ok and pred["abstained"]:
            err = "over_abstention"
        elif not ok:
            err = "generation_error"
        elif ok and not (gold <= set(pred["cited"])):
            err = "citation_gap"
        else:
            err = "ok"
        ERR.append({"qid": qid, "type": item.get("type"), "error": err,
                    "gold_in_context": float(gold <= ctx),
                    "abstained": float(pred["abstained"]),
                    "answered": float(not pred["abstained"]),
                    "correct": float(ok),
                    "n_cited": len(pred["cited"]),
                    "n_gold": len(gold)})
    ERR_DF = pd.DataFrame(ERR)
    ERR_DF.to_csv(ARTIFACT_DIR / "error_taxonomy.csv", index=False)

    print("\n=== hybrid error taxonomy (dev) ===")
    counts = ERR_DF["error"].value_counts()
    for k, v in counts.items():
        print(f"  {k:20s} {v:4d}  ({v / len(ERR_DF):5.1%})")

    # ---- write error examples ----
    def fmt_example(row, bucket):
        qid = row["qid"]
        item = QDEV_BY_QID[qid]
        label = LABELS_BY_QID[qid]
        pred = hybrid[qid]
        packed = CONTEXT_USED["hybrid"].get(qid, [])
        lines = [
            f"QID: {qid} | type: {item.get('type')} | bucket: {bucket}",
            f"question        : {item['question']}",
            f"gold answer     : {label['answer']} | aliases: {label.get('aliases', [])}",
            f"gold doc ids    : {gold_ids_of(qid)}",
            f"retrieved top10 : {RANKINGS['dev']['hybrid'][qid]}",
            f"context (packed): {packed}",
            f"system answer   : {pred['answer']!r} | abstained={pred['abstained']} | cited={pred['cited']}",
            f"raw output      : {RAW_OUTPUTS['hybrid'].get(qid, '')[:400]!r}",
        ]
        return "\n".join(lines)

    ERROR_TXT = OUTPUT_DIR / "error_analysis.txt"
    with ERROR_TXT.open("w", encoding="utf-8") as fh:
        fh.write("HYBRID RAG — ERROR ANALYSIS (dev)\n" + "=" * 60 + "\n\n")
        for bucket in ("retrieval_failure", "packing_evidence_gap", "over_abstention", "generation_error", "citation_gap"):
            subset = ERR_DF[ERR_DF.error == bucket].head(5)
            n_total = int((ERR_DF.error == bucket).sum())
            fh.write(f"\n### {bucket}  ({n_total} cases, {n_total/len(ERR_DF):.1%} of dev)\n\n")
            for _, row in subset.iterrows():
                fh.write(fmt_example(row, bucket) + "\n\n" + "-" * 50 + "\n\n")
    print("saved:", ERROR_TXT)

    print("\n=== sample retrieval failures ===")
    for _, row in ERR_DF[ERR_DF.error == "retrieval_failure"].head(3).iterrows():
        print(fmt_example(row, "retrieval_failure") + "\n")


=== metrics by question type ===


,system,type,n,recall_10,hitall_10,mrr,ndcg_10,em,f1,cite_precision,cite_recall,abstained
0,dense,comparison,73,100.0,100.0,97.9,97.4,74.0,86.6,64.4,63.0,1.4
1,dense,bridge_comparison,66,57.6,1.5,100.0,68.3,22.7,25.2,45.7,20.1,47.0
2,dense,compositional,125,75.2,50.4,99.6,76.8,12.8,19.2,34.0,17.6,64.0
3,dense,inference,36,90.3,80.6,95.8,82.5,19.4,43.2,76.9,50.0,8.3
4,hybrid,comparison,73,100.0,100.0,97.3,96.3,72.6,85.1,71.9,70.5,1.4
5,hybrid,bridge_comparison,66,59.1,7.6,99.2,68.3,24.2,26.2,46.2,21.2,50.0
6,hybrid,compositional,125,64.4,28.8,94.5,66.8,11.2,14.9,29.2,15.6,64.8
7,hybrid,inference,36,75.0,52.8,86.2,69.7,13.9,36.2,59.7,40.3,22.2
8,hybrid_sentence,comparison,73,100.0,100.0,97.9,96.5,74.0,84.9,67.1,66.4,2.7
9,hybrid_sentence,bridge_comparison,66,59.8,6.1,98.5,68.6,19.7,24.7,46.2,20.8,47.0



=== bridge_comparison bottleneck ===


,n,recall@2,recall@5,recall@10,hitall@2,hitall@5,hitall@10,mrr,em
system,,,,,,,,,
dense,66,47.7,55.7,57.6,0.0,0.0,1.5,100.0,22.7
hybrid,66,45.8,55.3,59.1,0.0,1.5,7.6,99.2,24.2
hybrid_sentence,66,45.8,56.1,59.8,0.0,1.5,6.1,98.5,19.7
hybrid_bge,66,48.9,57.2,58.7,0.0,4.5,4.5,99.2,27.3
hybrid_qwen,66,49.6,58.0,59.8,0.0,4.5,4.5,99.2,24.2
hybrid_goldmask,66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


average gold documents per bridge question: 4.00

=== hybrid error taxonomy (dev) ===
  retrieval_failure     167  (55.7%)
  generation_error       42  (14.0%)
  ok                     37  (12.3%)
  citation_gap           28  ( 9.3%)
  packing_evidence_gap   22  ( 7.3%)
  over_abstention         4  ( 1.3%)
saved: /kaggle/working/NLP_phase2/error_analysis.txt

=== sample retrieval failures ===
QID: q0002 | type: bridge_comparison | bucket: retrieval_failure
question        : Which film has the director who was born later, The Third Side Of The River or Escape From L.A.?
gold answer     : The Third Side Of The River | aliases: ['The Third Side of the River', 'Third Side of the River']
gold doc ids    : ['p03489', 'p01066', 'p00620', 'p01750']
retrieved top10 : ['p03489', 'p01066', 'p01068', 'p03449', 'p00835', 'p02156', 'p03756', 'p00788', 'p01883', 'p03223']
context (packed): ['p03489', 'p01066', 'p01068', 'p03449', 'p00835', 'p02156']
system answer   : '' | abstained=True | cited=[]
ra

## §14 Hidden split

Generate hybrid predictions and validate their structure with `score.py --validate-only`. No hidden labels or hidden accuracy are assumed.


In [17]:
# =============================================================================
# §14  Run final systems on the hidden split and validate outputs
# =============================================================================
HIDDEN_PREDS = {}
HIDDEN_SYSTEMS = [s for s in CONFIG["hidden_systems"] if s in RANKINGS["hidden"]]
assert HIDDEN_SYSTEMS, "No systems selected for the hidden split (check CONFIG['hidden_systems'])"

for system in HIDDEN_SYSTEMS:
    print(f"\n=== hidden / {system} ===")
    HIDDEN_PREDS[system] = run_system(system, "hidden", questions_hidden,
                                      RANKINGS["hidden"][system])

for system in HIDDEN_SYSTEMS:
    path = pred_path(system, "hidden")
    print(f"\nvalidating {path}")
    print(run_score(path, "hidden", validate_only=True))
    rows = read_jsonl(path)
    assert len(rows) == len(questions_hidden), (
        f"expected {len(questions_hidden)} hidden predictions, got {len(rows)}"
    )
    assert {r["qid"] for r in rows} == {str(q["qid"]) for q in questions_hidden}, (
        "hidden qid set does not match questions_hidden.jsonl"
    )
    assert set(rows[0]) == REQUIRED_FIELDS, (
        f"hidden predictions must contain exactly fields {REQUIRED_FIELDS}, got {set(rows[0])}"
    )
print("hidden predictions validated.")



=== hidden / hybrid ===
  [hybrid] 50/300  (198s, ETA 992s)
  [hybrid] 100/300  (410s, ETA 820s)
  [hybrid] 150/300  (616s, ETA 616s)
  [hybrid] 200/300  (821s, ETA 411s)
  [hybrid] 250/300  (1024s, ETA 205s)
  [hybrid] 300/300  (1237s, ETA 0s)
Saved 300 predictions -> /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl
  saved 300 predictions -> /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl

validating /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl
valid: /kaggle/working/NLP_phase2/runs/hidden/hybrid/predictions.jsonl (300 predictions)

hidden predictions validated.


## §15.1 Figures

Figures use this run's score and gold-mask analysis outputs. Missing metrics are not treated as zero.


In [18]:
# =============================================================================
# §15-1  Figures
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIGS = {}

if CONFIG["run_dev"] and SCORES:
    show_systems = [s for s in ("phase1_dense", "dense", "hybrid",
                                "hybrid_sentence", "hybrid_bge", "hybrid_qwen") if s in SCORES]
    metrics = [("em", "EM"), ("f1", "F1"), ("hitall@10", "HitAll@10"),
               ("recall@10", "Recall@10"), ("cite_precision", "Cite-P")]
    metrics = [(m, lab) for m, lab in metrics if all(m in SCORES[s]["overall"] for s in show_systems)]

    # ---- Figure 1: overall comparison across key metrics ----
    fig, ax = plt.subplots(figsize=(7.4, 3.2), constrained_layout=True)
    x = np.arange(len(metrics)); w = 0.8 / max(len(show_systems), 1)
    for i, system in enumerate(show_systems):
        vals = [SCORES[system]["overall"][m]["value"] for m, _ in metrics]
        bars = ax.bar(x + (i - (len(show_systems) - 1) / 2) * w, vals, w, label=system)
        ax.bar_label(bars, fmt="%.0f", fontsize=6, padding=1)
    ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in metrics], fontsize=8)
    ax.set_ylim(0, 105); ax.set_ylabel("score (%)", fontsize=8)
    ax.legend(fontsize=7, ncol=len(show_systems)); ax.tick_params(labelsize=7)
    ax.set_title("Dev overall: retrieval variants (score.py)", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    FIGS["overall"] = FIG_DIR / "fig_overall.png"
    fig.savefig(FIGS["overall"], dpi=200); plt.close(fig)

    # ---- Figure 2: per-question-type (EM and HitAll@10) ----
    types = [t for t in TYPE_ORDER if t in SCORES["hybrid"]["by_type"]]
    if types:
        fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9), constrained_layout=True)
        for ax, metric, title in ((axes[0], "hitall@10", "HitAll@10 by type"),
                                  (axes[1], "em", "EM by type")):
            x = np.arange(len(types)); w = 0.8 / max(len(show_systems), 1)
            for i, system in enumerate(show_systems):
                vals = [SCORES[system]["by_type"].get(t, {}).get(metric, {}).get("value", np.nan)
                        for t in types]
                bars = ax.bar(x + (i - (len(show_systems) - 1) / 2) * w, vals, w,
                              label=system)
                ax.bar_label(bars, fmt="%.0f", fontsize=5, padding=1)
            ax.set_xticks(x)
            ax.set_xticklabels([t.replace("_", "\n") for t in types], fontsize=6.5)
            ax.set_ylim(0, 105); ax.set_title(title, fontsize=9)
            ax.tick_params(labelsize=7); ax.spines[["top", "right"]].set_visible(False)
        axes[0].legend(fontsize=6, ncol=2)
        FIGS["by_type"] = FIG_DIR / "fig_by_type.png"
        fig.savefig(FIGS["by_type"], dpi=200); plt.close(fig)

    # ---- Figure 3: abstention experiment ----
    if ABST_SUMMARY.get("category_counts"):
        cats = ["masked_abstained", "masked_correct", "masked_wrong"]
        vals = [ABST_SUMMARY["category_counts"].get(c, 0) for c in cats]
        order = np.argsort(vals)[::-1]
        fig, ax = plt.subplots(figsize=(6.4, 2.6), constrained_layout=True)
        bars = ax.bar([cats[i] for i in order], [vals[i] for i in order], color="#4C72B0")
        ax.bar_label(bars, fontsize=7)
        ax.set_title("Gold-masked abstention experiment (dev, n=%d)" % ABST_SUMMARY["n"], fontsize=9)
        ax.tick_params(labelsize=7, axis="x", rotation=18)
        ax.spines[["top", "right"]].set_visible(False)
        FIGS["abstention"] = FIG_DIR / "fig_abstention.png"
        fig.savefig(FIGS["abstention"], dpi=200); plt.close(fig)

print("figures:", {k: str(v) for k, v in FIGS.items()})


figures: {'overall': '/kaggle/working/NLP_phase2/figures/fig_overall.png', 'by_type': '/kaggle/working/NLP_phase2/figures/fig_by_type.png', 'abstention': '/kaggle/working/NLP_phase2/figures/fig_abstention.png'}


In [19]:
# =============================================================================
# §15-2  Evidence-based PDF report (maximum three pages) and submission archive
# =============================================================================
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.enums import TA_RIGHT, TA_CENTER
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, Flowable
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from xml.sax.saxutils import escape
from pypdf import PdfReader
import arabic_reshaper
from bidi.algorithm import get_display

if not CONFIG["run_dev"] or not {"hybrid", "phase1_dense"} <= set(SCORES):
    raise RuntimeError("A full dev evaluation with the Phase-1 baseline is required for the report.")
if not {"hybrid", "hybrid_sentence", "hybrid_goldmask"} <= set(DEV_PREDS):
    raise RuntimeError("Run hybrid, sentence-level hybrid and gold-mask experiments before creating the report.")
if CONFIG["dev_limit"] is not None:
    raise RuntimeError("Partial dev runs are for testing only; set dev_limit=None before submission.")
if GITHUB_COMMIT == "YOUR_COMMIT_HASH":
    print("WARNING: Set GITHUB_COMMIT to the actual submission commit, then rebuild the report before submission.")

font_dir = OUTPUT_DIR / "fonts"
font_dir.mkdir(parents=True, exist_ok=True)
font_path = font_dir / "Vazirmatn-Regular.ttf"
if not font_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(
            "https://cdn.jsdelivr.net/gh/rastikerdar/vazirmatn@v33.003/fonts/ttf/Vazirmatn-Regular.ttf",
            font_path)
    except Exception as exc:
        raise RuntimeError("Persian report font unavailable; add Vazirmatn-Regular.ttf or enable internet.") from exc
pdfmetrics.registerFont(TTFont("Vazir", str(font_path)))

def fa(text):
    return escape(get_display(arabic_reshaper.reshape(str(text))))

s_title = ParagraphStyle("p2title", fontName="Vazir", fontSize=13, leading=19, alignment=TA_CENTER, spaceAfter=6)
s_head = ParagraphStyle("p2head", fontName="Vazir", fontSize=10, leading=15, alignment=TA_RIGHT, spaceBefore=5, spaceAfter=3, textColor=colors.HexColor("#204878"))
s_body = ParagraphStyle("p2body", fontName="Vazir", fontSize=8.1, leading=12.5, alignment=TA_RIGHT, spaceAfter=4)
s_small = ParagraphStyle("p2small", fontName="Vazir", fontSize=7.1, leading=10.5, alignment=TA_RIGHT, spaceAfter=3)
s_table = ParagraphStyle("p2table", fontName="Vazir", fontSize=6.7, leading=9, alignment=TA_CENTER)

class RTLText(Flowable):
    """Wrap logical text first, then apply Persian shaping and bidi per line."""
    def __init__(self, text, style, bar=False):
        super().__init__()
        self.text, self.style, self.bar = str(text), style, bar
        self.spaceAfter = style.spaceAfter
        self.spaceBefore = style.spaceBefore

    def _visual(self, text):
        return get_display(arabic_reshaper.reshape(text))

    def wrap(self, avail_width, avail_height):
        self.width = avail_width
        inset = 10 if self.bar else 0
        usable = max(1, avail_width - inset)
        size = self.style.fontSize
        words = self.text.split()
        lines, line = [], ""
        for word in words:
            candidate = (line + " " + word).strip()
            width = pdfmetrics.stringWidth(self._visual(candidate), "Vazir", size)
            if line and width > usable:
                lines.append(line)
                line = word
            else:
                line = candidate
        if line or not lines:
            lines.append(line)
        self.lines = lines
        self.height = len(lines) * self.style.leading + (5 if self.bar else 0)
        return self.width, self.height

    def draw(self):
        c = self.canv
        if self.bar:
            c.setFillColor(colors.HexColor("#eff3f9"))
            c.roundRect(0, 0, self.width, self.height, 3, fill=1, stroke=0)
            c.setFillColor(colors.HexColor("#167ad0"))
            c.rect(self.width - 3, 0, 3, self.height, fill=1, stroke=0)
        c.setFont("Vazir", self.style.fontSize)
        c.setFillColor(self.style.textColor or colors.black)
        for i, line in enumerate(self.lines):
            visual = self._visual(line)
            y = self.height - self.style.leading * (i + 1) + 2
            if self.style.alignment == TA_CENTER:
                c.drawCentredString(self.width / 2, y, visual)
            else:
                c.drawRightString(self.width - (7 if self.bar else 0), y, visual)

def para(text, style=s_body):
    return RTLText(text, style, bar=(style is s_head))

s_meta_label = ParagraphStyle("metalabel", fontName="Vazir", fontSize=7.8, leading=11, alignment=TA_RIGHT, textColor=colors.white)
s_meta_value = ParagraphStyle("metavalue", fontName="Vazir", fontSize=7.4, leading=10, textColor=colors.white)

def header_card():
    title_style = ParagraphStyle("cardtitle", fontName="Vazir", fontSize=11.4, leading=17,
                                 alignment=TA_CENTER, textColor=colors.white)
    title = Table([[RTLText("گزارش فنی فاز دوم: بازیابی ترکیبی و پاسخ مستند", title_style)]], colWidths=[17.3*cm])
    title.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,-1), colors.HexColor("#172c58")),
                               ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 5)]))
    meta = [
        ("تهیه‌کننده", STUDENT_NAME, True),
        ("شماره دانشجویی", STD_ID, False),
        ("مخزن GitHub", GITHUB_URL, False),
        ("شناسه Commit", GITHUB_COMMIT, False),
        ("مدل Embedding", CONFIG["embedding_model"], False),
        ("مدل تولید پاسخ", CONFIG["llm_model"], False),
        ("سخت‌افزار", "دو GPU از نوع Tesla T4 در محیط Kaggle، با دقت float16", True),
        ("تاریخ اعلام‌شده", REPORT_DATE, False),
    ]
    cells = []
    for label, value, is_fa in meta:
        v = RTLText(value, s_meta_label) if is_fa else Paragraph(escape(str(value)), s_meta_value)
        cells.append([v, RTLText(label, s_meta_label)])
    t = Table(cells, colWidths=[13.2*cm, 4.1*cm], hAlign="CENTER")
    t.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,-1), colors.HexColor("#172c58")),
        ("VALIGN",(0,0),(-1,-1),"MIDDLE"), ("RIGHTPADDING",(0,0),(-1,-1),8),
        ("LEFTPADDING",(0,0),(-1,-1),8), ("TOPPADDING",(0,0),(-1,-1),1),
        ("BOTTOMPADDING",(0,0),(-1,-1),1),
        ("LINEAFTER", (0,0), (0,-1), .3, colors.HexColor("#52709e"))]))
    return [title, t, Spacer(1, 5)]

def table(headers, rows, widths):
    cells = [[Paragraph(escape(str(x)), s_table) for x in headers]]
    cells += [[Paragraph(escape(str(x)), s_table) for x in row] for row in rows]
    t = Table(cells, colWidths=widths, repeatRows=1, hAlign="CENTER")
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#dce9f4")),
        ("GRID", (0, 0), (-1, -1), .25, colors.HexColor("#a5b2bd")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 2), ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ("LEFTPADDING", (0, 0), (-1, -1), 2), ("RIGHTPADDING", (0, 0), (-1, -1), 2),
    ]))
    return t

def metric(system, name, qtype=None, interval=False):
    scores = SCORES.get(system, {})
    part = scores.get("overall", {}) if qtype is None else scores.get("by_type", {}).get(qtype, {})
    v = part.get(name)
    if v is None:
        return "N/A"
    if interval:
        return f"{v['value']:.1f} [{v['ci'][0]:.1f},{v['ci'][1]:.1f}]"
    return f"{v['value']:.1f}"

def paired(entry):
    if entry is None:
        return "N/A"
    return f"{entry['value']:+.1f} [{entry['ci'][0]:+.1f},{entry['ci'][1]:+.1f}]"

def fig(path, width, height):
    return Image(str(path), width=width * cm, height=height * cm)

systems = [x for x in ("phase1_dense", "dense", "hybrid", "hybrid_sentence", "hybrid_bge", "hybrid_qwen", "hybrid_goldmask") if x in SCORES]
n = SCORES["hybrid"]["overall_n"]
if n != len(questions_dev):
    raise RuntimeError("Report cannot be submitted: dev scores do not cover all dev questions.")
for system in systems:
    if SCORES[system]["overall_n"] != n:
        raise RuntimeError(f"Mismatched dev count for {system}.")

story = header_card() + [
    para("۱. داده، روش و تنظیمات", s_head),
    para(f"ارزیابی روی {n} پرسش dev و {len(DOC_IDS)} سند انجام شده است. دادهٔ hidden برچسب عمومی ندارد و تنها اعتبار ساختاری خروجی آن بررسی می‌شود. مدل تعبیه {CONFIG['embedding_model']} و مدل مولد {CONFIG['llm_model']} است."),
    para(f"خروجی hidden شامل {len(HIDDEN_PREDS.get('hybrid', []))} پیش‌بینی hybrid است و با score.py از نظر ساختاری اعتبارسنجی شده است. دو بازرتبه‌بند BGE و Qwen 0.6B در dev با نام‌های hybrid_bge و hybrid_qwen مقایسه می‌شوند.", s_small),
    para(f"بازیابی متراکم بر embedding بارگذاری‌شده یا ساخته‌شده با تابع تعبیهٔ فاز اول تکیه دارد. BM25 دست‌نویس با k1={CONFIG['bm25_k1']} و b={CONFIG['bm25_b']}، سپس RRF با ثابت {CONFIG['rrf_k']} ترکیب می‌شود. تا {CONFIG['k_context']} سند با سقف {CONFIG['max_context_tokens']} توکن مدل بسته‌بندی می‌شوند. شناسهٔ ارجاعات به اسناد حاضر در متن محدود می‌شود؛ مدل می‌تواند امتناع کند."),
    para("۲. نتایج کلی dev", s_head)]
rows = [[x] + [metric(x, m) for m in ("recall@10", "hitall@10", "mrr", "ndcg@10", "em", "f1", "cite_precision", "cite_recall", "abstained")] for x in systems]
story.append(table(["system", "R@10", "H@10", "MRR", "nDCG", "EM", "F1", "Cite-P", "Cite-R", "Abst"], rows, [3.5*cm]+[1.5*cm]*9))
story.append(para("همهٔ اعداد جدول درصد هستند. dense مبنای کنترل‌شده و phase1_dense خروجی اصلی ذخیره‌شدهٔ فاز اول است. کنترل BM25 با rank_bm25 پس از یکسان‌سازی فرمول IDF و آزمون واژه‌های ناشناخته اجرا شد.", s_small))
if "overall" in FIGS:
    story.append(fig(FIGS["overall"], 14.3, 5.4))
story += [PageBreak(), para("۳. مقایسهٔ جفتی و تفکیک نوع پرسش", s_head)]
key = "hybrid__vs__dense"
pair = PAIRED.get(key, {})
if not pair:
    raise RuntimeError("Missing controlled paired comparison with the dense baseline.")
story.append(para("مقایسهٔ اصلی hybrid منهای dense کنترل‌شده است: هر دو از مدل، پرامپت، parser، بودجهٔ زمینه و تنظیمات تولید فاز دوم استفاده می‌کنند و بازیاب فرق دارد. dense با همان بردارهای فعلی بازسازی شده است؛ ممکن است رتبه‌بندی آن با خروجی ذخیره‌شدهٔ فاز اول متفاوت باشد.", s_small))
story.append(table(["Metric", "Hybrid - controlled dense [95% CI]"], [[m, paired(pair.get(m))] for m in ("hitall@10", "em", "f1", "cite_precision", "cite_recall")], [5*cm, 11*cm]))
_historical_pair = PAIRED.get("hybrid__vs__phase1_dense", {})
story.append(para(f"مقایسه با پاسخ ذخیره‌شدهٔ اصلی فاز اول، تاریخی است و به علت تفاوت احتمالی prompt و تنظیمات منصفانهٔ کنترل‌شده نیست. ترتیب ده سند dense بازسازی‌شده در {DENSE_RANK_AUDIT['exact_order_percent']:.1f} درصد پرسش‌ها با خروجی اصلی یکسان بود.", s_small))
if _historical_pair:
    story.append(table(["Metric", "Hybrid - original Phase 1 [95% CI]"],
        [[m, paired(_historical_pair.get(m))] for m in ("hitall@10", "em")], [5*cm, 11*cm]))
variant_rows = [[name, paired(PAIRED[name].get("hitall@10")), paired(PAIRED[name].get("em")), paired(PAIRED[name].get("f1"))]
                for name in ("hybrid_sentence__vs__hybrid", "hybrid_bge__vs__hybrid", "hybrid_qwen__vs__hybrid", "hybrid_qwen__vs__hybrid_bge") if name in PAIRED]
if variant_rows:
    story.append(table(["variant difference", "H@10 [95% CI]", "EM [95% CI]", "F1 [95% CI]"], variant_rows, [5*cm, 4*cm, 4*cm, 4*cm]))
if CONFIG["run_rerank"]:
    _bge_t = RERANK_TIMINGS["bge"]
    _qwen_t = RERANK_TIMINGS["qwen"]
    story.append(para(f"دو بازرتبه‌بند دقیقاً کاندیداهای یکسان hybrid را رتبه‌بندی کردند. زمان امتیازدهی BGE برابر {_bge_t['scoring_seconds']:.1f} ثانیه و Qwen برابر {_qwen_t['scoring_seconds']:.1f} ثانیه است؛ بارگذاری و تولید پاسخ در این زمان‌ها نیست. tokenizer و قالب دو مدل متفاوت‌اند، بنابراین برش متن دقیقاً یکسان نیست. انتخاب بر اساس dev و بررسی دقت و هزینه انجام شود، نه بر اساس hidden.", s_small))
story.append(para("تفاضل به تفکیک نوع پرسش در جدول بعد نسبت به dense کنترل‌شده است.", s_small))
ptype = []
for t in sorted(PAIRED_BY_TYPE):
    d = PAIRED_BY_TYPE[t]
    ptype.append([t, str(next(iter(d.values()))["n"]), paired(d.get("hitall@10")), paired(d.get("em"))])
story.append(table(["type", "n", "H@10 delta [95% CI]", "EM delta [95% CI]"], ptype, [4.2*cm, 1*cm, 5.4*cm, 5.4*cm]))
if ptype:
    best = max(PAIRED_BY_TYPE, key=lambda t: PAIRED_BY_TYPE[t].get("hitall@10", {}).get("value", float("-inf")))
    story.append(para(f"بیشترین تفاضل مشاهده‌شدهٔ HitAll@10 در دستهٔ {best} است. چرایی این تفاوت صرفاً با این آمار ثابت نمی‌شود؛ نمونه‌های بازیابی در فایل تحلیل خطا برای بررسی شواهد ارائه شده‌اند.", s_small))
if "by_type" in FIGS:
    story.append(fig(FIGS["by_type"], 14.3, 3.9))
story += [PageBreak(), para("۴. دانه‌بندی و امتناع با ماسک اسناد طلایی", s_head)]
story.append(para("در آزمایش دانه‌بندی، امتیاز جمله‌های هر سند با بیشینه تجمیع و در سطح سند گزارش می‌شود. در gold-mask همهٔ اسناد طلایی پیش از top-k در هر دو بازیاب کنار گذاشته می‌شوند. درست‌بودن پاسخ بدون سند طلایی لزوماً اثبات دانش پارامتری نیست: ممکن است شواهد دیگری در مجموعه وجود داشته باشد.", s_small))
maskrows = [["masked abstained", f"{100*ABST_SUMMARY['masked_abstention_rate']:.1f}%"], ["masked correct", f"{100*ABST_SUMMARY['masked_correct_rate']:.1f}%"], ["masked wrong", f"{100*ABST_SUMMARY['masked_wrong_rate']:.1f}%"], ["normal over-abstention", f"{100*ABST_SUMMARY['over_abstention_rate']:.1f}%"]]
story.append(table(["Observed outcome", "All dev queries"], maskrows, [8*cm, 8*cm]))
story.append(para("در اجرای ماسک‌شده صفر شدن معیارهای بازیابی و ارجاع به سند طلایی قابل انتظار است، چون اسناد طلایی عمداً حذف شده‌اند. سه ردیف اول نتایج جداگانهٔ همان اجرا هستند و ردیف آخر مستقل و مربوط به اجرای عادی است. پاسخ نادرست لزوماً اثبات نبود شواهد جایگزین نیست.", s_small))
if "abstention" in FIGS:
    story.append(fig(FIGS["abstention"], 13.0, 4.4))
story.append(para("۵. خطاها، محدودیت‌ها و بازتولیدپذیری", s_head))
error_counts = ERR_DF["error"].value_counts().to_dict()
story.append(table(["Observed error category", "Count"], [[k, str(v)] for k, v in sorted(error_counts.items())], [11*cm, 5*cm]))
story.append(para("دسته‌بندی خطا بر اساس بازیابی top-10، اسناد واقعاً بسته‌بندی‌شده، پاسخ و ارجاع محاسبه شده است. جزئیات نمونه‌ها و متن خام در error_analysis.txt و artifacts ذخیره می‌شوند. معیارهای استناد، تطابق شناسه‌ها با اسناد طلایی‌اند، نه داوری معنایی صحت هر ادعا.", s_small))
story.append(para("مقایسهٔ جمله و بازرتبه‌بندی در جداول و مقایسه‌های جفتی ذخیره شده است. خطای عددی، محدودیت بودجهٔ توکن، متفاوت‌بودن پرامپت مبنا و محدودبودن ارزیابی به dev از محدودیت‌ها هستند. از دستیار هوش مصنوعی برای توسعه و بازبینی کد و گزارش استفاده شده است؛ نتایج عددی صرفاً از اجرای نوت‌بوک و score.py به دست می‌آیند.", s_small))
REPORT_PATH = OUTPUT_DIR / "report.pdf"
doc = SimpleDocTemplate(str(REPORT_PATH), pagesize=A4, leftMargin=1.2*cm, rightMargin=1.2*cm, topMargin=0.9*cm, bottomMargin=0.9*cm, title="Phase 2 report", author=STUDENT_NAME)
def _page_footer(canvas, document):
    canvas.saveState()
    canvas.setFont("Vazir", 7)
    canvas.setFillColor(colors.HexColor("#71809b"))
    canvas.drawCentredString(A4[0] / 2, .5 * cm, get_display(arabic_reshaper.reshape(f"صفحه {document.page}")))
    canvas.restoreState()

doc.build(story, onFirstPage=_page_footer, onLaterPages=_page_footer)
n_pages = len(PdfReader(str(REPORT_PATH)).pages)
if n_pages > 3:
    REPORT_PATH.unlink(missing_ok=True)
    raise RuntimeError(f"Report is {n_pages} pages; the project limit is 3 pages. Reduce report content before submission.")
print(f"report built: {REPORT_PATH} ({n_pages} pages)")

SUBMISSION_ZIP = Path("/kaggle/working") / f"P2_{STD_ID}.zip"
if not SUBMISSION_ZIP.parent.exists():
    SUBMISSION_ZIP = OUTPUT_DIR / f"P2_{STD_ID}.zip"
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(REPORT_PATH, arcname="report.pdf")
    zf.write(PROMPT_FILE, arcname="prompts/phase2_strict.txt")
    _embedding_for_zip = Path(_p1_embedding_path) if _p1_embedding_path is not None else PASSAGE_EMB_PATH
    if not _embedding_for_zip.is_file():
        raise FileNotFoundError(f"The passage embedding file is missing: {_embedding_for_zip}")
    zf.write(_embedding_for_zip, arcname="embeddings/document_embeddings_4B.npz", compress_type=zipfile.ZIP_STORED)
    if _p1_embedding_path is None and PASSAGE_META_PATH.is_file():
        zf.write(PASSAGE_META_PATH, arcname="embeddings/document_embeddings_4B.meta.json")
    if ERROR_TXT.exists():
        zf.write(ERROR_TXT, arcname="analysis/error_analysis.txt")
    for path in sorted(ARTIFACT_DIR.glob("*.json")) + sorted(ARTIFACT_DIR.glob("*.csv")) + sorted(ARTIFACT_DIR.glob("score_*.txt")) + sorted(ARTIFACT_DIR.glob("paired_*.txt")):
        zf.write(path, arcname=f"analysis/{path.name}")
    for path in sorted(FIG_DIR.glob("*.png")):
        zf.write(path, arcname=f"figures/{path.name}")
    for split in ("dev", "hidden"):
        base = RUNS_DIR / split
        if base.exists():
            for run_dir in sorted(p for p in base.iterdir() if p.is_dir()):
                for name in ("predictions.jsonl", "run_config.yaml"):
                    fp = run_dir / name
                    if fp.exists():
                        zf.write(fp, arcname=str(fp.relative_to(OUTPUT_DIR)))
with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    assert "report.pdf" in names
    assert "embeddings/document_embeddings_4B.npz" in names
    assert sum(n.endswith("predictions.jsonl") for n in names) == sum(n.endswith("run_config.yaml") for n in names)
print("submission:", SUBMISSION_ZIP)


report built: /kaggle/working/NLP_phase2/report.pdf (3 pages)
submission: /kaggle/working/P2_404722145.zip


## Submission checklist

- Save `artifacts/document_embeddings_4B.npz` from the Kaggle output for future sessions; upload it to the input dataset to reuse it.
- Fill in the real `GITHUB_COMMIT` in CONFIG, run all cells, and inspect `report.pdf`.
- Use `dev_limit=None` for submission; check hidden validation and the archive.
- `P2_<StudentID>.zip` contains the report, run files, figures and analysis artifacts. The PDF builder rejects reports over three pages.
